# 04 · Modelagem

**CRISP-DM fase 4 — modelagem** · Meridiano Imobiliária

---

O notebook [`03-preparacao`](03-preparacao.ipynb) entregou duas coisas e nada além delas: a
**base de modelagem** (2.350 imóveis, 24 colunas, em unidades de negócio) e a **receita de
pré-processamento** não ajustada. Não entregou *k*, não entregou algoritmo e não entregou
segmento — de propósito, e está escrito lá.

Este notebook resolve as quatro tarefas que a fase 4 do CRISP-DM define, na ordem em que o
método as define, porque a ordem é o argumento: **escolher a técnica antes de rodá-la** é o
que separa uma decisão de modelagem de uma justificativa escrita depois do resultado.

| Bloco | Tarefa do CRISP-DM | Pergunta | O que decide |
|:--|:---|:---|:---|
| 1 | *Select modeling technique* | Que algoritmos podem responder à pergunta? | A premissa de cada família, confrontada com a medida na base |
| 2 | *Generate test design* | Como se testa um agrupamento, se não há rótulo? | Reamostragem, *hold-out* e critérios de aceitação declarados **antes** |
| 3 | *Build model* | Quantos grupos, com que parâmetro? | A varredura, a estabilidade e o teto de cinco linhas de produto |
| 4 | *Assess model* | Qual modelo segue adiante? | O confronto contra os critérios do bloco 2, sem exceção aberta depois |

A regra deste notebook é uma só:

> **O critério vem antes do resultado.** Todo número de aceitação usado no bloco 4 está
> escrito no bloco 2, e o bloco 2 roda antes de qualquer algoritmo ver a carteira. Critério
> ajustado depois de ver a saída não é critério, é justificativa.

## O que este notebook não faz

Três coisas ficam de fora, e nenhuma por esquecimento:

* **não abre `preco_venda` nem `bairro`.** A avaliação daqui é **técnica** — coesão,
  estabilidade, generalização. A leitura de negócio, o confronto com o preço e a validação
  externa por bairro são a fase 5, e usá-los agora seria escolher o modelo pela resposta;
* **não batiza segmento nem escreve persona.** Nomear um grupo é traduzir estatística em
  política comercial, e isso pressupõe o modelo já escolhido;
* **não reabre decisões da fase 3.** O espaço de 17 variáveis, a padronização e a recusa do
  PCA foram medidos lá. Aqui eles são **entrada**, e a única coisa que este notebook faz com
  eles é conferir que chegaram inteiros.

In [ ]:
# Importações
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from joblib import dump, load
from scipy.spatial.distance import pdist
from sklearn.base import clone
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_samples, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.pipeline import Pipeline

# O notebook roda a partir de `notebooks/`; a raiz entra no path para que
# `import src...` funcione sem instalar o pacote.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src import config
from src.data import dicionario as dic
from src.model import densidade as den
from src.model import distancias as dist
from src.model import estabilidade as estab
from src.model import hierarquica as hier
from src.model import mistura as mist
from src.model import particional as part
from src.model import perfilagem as perf
from src.model import personas as pers
from src.utils import io

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

io.configurar_estilo()

# As figuras deste notebook vão para `docs/imagens/modelagem/`; as tabelas, para
# `reports/tables/modelagem/`. A regra do projeto é que nenhuma figura existe sem
# a tabela que a produziu.
FIGURAS = config.DOCS / "imagens" / "modelagem"
FIGURAS.mkdir(parents=True, exist_ok=True)
SECAO = "modelagem"

# Cores da identidade do projeto (docs/assets/css/brand.css).
BARRA = config.AZUL_COBALTO
REFERENCIA = config.AMARELO
DESTAQUE = config.AZUL_CLARO
ALERTA = config.CORAL
NEUTRO = config.CINZA
APROVADO = config.VERDE


def salvar(fig, nome, dados=None):
    """Grava a figura em `docs/imagens/modelagem/` e a tabela que a originou."""
    fig.savefig(FIGURAS / f"{nome}.png", bbox_inches="tight")
    plt.close(fig)
    if dados is not None:
        io.salvar_tabela(dados, SECAO, nome, indice=False)


print(f"Projeto em {RAIZ}")
print(f"Semente do projeto: {config.SEMENTE}")

## O registro de decisões

O notebook 03 registrou *decisões de preparação*. Este registra **decisões de modelagem**, e
elas não são todas do mesmo tipo:

| `acao` | Significado |
|:---|:---|
| `técnica selecionada` | A família entra no comparativo, e a premissa que a sustenta está medida |
| `técnica descartada` | Não entra, e o motivo é a premissa violada — não a preferência |
| `critério declarado` | Regra de aceitação escrita **antes** de rodar qualquer algoritmo |
| `parâmetro fixado` | *k*, `eps`, ligação, covariância: o valor e o número que o escolheu |
| `modelo reprovado` | Foi construído, medido e não passou — com a medida ao lado |
| `modelo escolhido` | O que segue para a fase 5 |

A distinção entre `técnica descartada` e `modelo reprovado` é a que mais importa numa banca.
A primeira é decisão *a priori*, por premissa; a segunda é resultado *a posteriori*, por
medida. Dizer que um algoritmo "não funcionaria" quando ele foi rodado e perdeu é trocar uma
pela outra.

In [ ]:
DECISOES: list[dict] = []


def pt(numero) -> str:
    """Formata o número no padrão brasileiro: ponto como separador de milhar."""
    return f"{numero:,}".replace(",", ".")


def registrar(
    bloco: str,
    decisao: str,
    tecnica: str,
    motivo: str,
    evidencia: str,
    acao: str = "técnica selecionada",
):
    """Anota uma decisão de modelagem, com a técnica, o motivo e a medida.

    `evidencia` guarda o **número** que sustenta a decisão. Decisão de modelagem
    sem número ao lado é opinião, e opinião não sobrevive a uma reexecução.
    """
    DECISOES.append(
        {
            "n": len(DECISOES) + 1,
            "bloco": bloco,
            "decisao": decisao,
            "tecnica": tecnica,
            "motivo": motivo,
            "evidencia": evidencia,
            "acao": acao,
        }
    )
    print(f"{len(DECISOES):>2}. [{acao:>19}] {decisao} — {evidencia}")

# 0 · A entrada — a base de modelagem e a receita

> `data/processed/imoveis_curados.parquet` **não** é lido aqui. A base curada tem o preço e
> o bairro; a base de modelagem não tem. Abrir a curada neste notebook seria abrir mão da
> garantia física que a fase 3 construiu.

Três conferências antes de qualquer algoritmo, e nenhuma é decorativa:

1. **o formato bate com o manifesto da preparação** — se o notebook 03 rodar de novo com
   outro limiar, o espaço muda e as decisões daqui mudam junto;
2. **o rótulo não está no arquivo** — nem `preco_venda`, nem `preco_por_pe2`, nem `bairro`.
   A conferência é redundante com a que a fase 3 já fez, e é por isso mesmo que ela fica: as
   duas pontas do vazamento têm de falhar ao mesmo tempo para o erro passar;
3. **o pré-processador chegou não ajustado** — se ele viesse de disco já com médias e
   desvios, todo o desenho de experimento do bloco 2 estaria comprometido antes de começar.

In [ ]:
BASE_MODELAGEM = config.DATA_PROCESSED / "base_modelagem.parquet"
PREPROCESSADOR = config.MODELS / "preprocessador-euclidiano.joblib"

base = pd.read_parquet(BASE_MODELAGEM)
manifesto_preparacao = io.ler_json(config.DATA_PROCESSED / "manifesto_preparacao.json")
preprocessador = load(PREPROCESSADOR)

ESPACO_EUCLIDIANO = manifesto_preparacao["espaco_euclidiano"]
ESPACO_MISTO = manifesto_preparacao["espaco_misto"]
IDENTIFICADORES = manifesto_preparacao["identificadores"]
LINHAS = len(base)

PROIBIDAS = {"preco_venda", "preco_por_pe2", "bairro"}
assert not PROIBIDAS & set(base.columns), "rótulo ou reservada entrou na base de modelagem"
assert LINHAS == manifesto_preparacao["linhas"], "linhas divergem do manifesto da preparação"
assert list(base.columns) == IDENTIFICADORES + ESPACO_EUCLIDIANO + ESPACO_MISTO, (
    "o esquema da base de modelagem mudou"
)
assert int(base.isna().sum().sum()) == 0, "a base de modelagem não pode ter célula vazia"

# Um `ColumnTransformer` ajustado ganha o atributo `transformers_`. A receita
# precisa chegar crua, e isso é conferido, não presumido.
assert not hasattr(preprocessador, "transformers_"), "o pré-processador chegou já ajustado"

entrada = pd.Series(
    {
        "arquivo": BASE_MODELAGEM.relative_to(RAIZ).as_posix(),
        "imóveis": pt(LINHAS),
        "espaço euclidiano": f"{len(ESPACO_EUCLIDIANO)} variáveis",
        "espaço misto": f"{len(ESPACO_MISTO)} nominais, em unidade original",
        "transformações de forma": manifesto_preparacao["transformacoes_de_forma"],
        "escalador da receita": manifesto_preparacao["escalador"],
        "escala aplicada no arquivo": manifesto_preparacao["escala_aplicada_no_arquivo"],
        "reservadas fora do arquivo": ", ".join(
            manifesto_preparacao["reservadas_fora_do_arquivo"]
        ),
        "decisões herdadas": manifesto_preparacao["decisoes"],
    }
)
display(entrada.to_frame("entrada deste notebook"))
base.head(3)

In [ ]:
# O ajuste aqui é para **diagnóstico**. O bloco 2 declara que toda medida de
# validação reajusta a receita dentro da sua própria amostra.
EUCLIDIANO = preprocessador.fit_transform(base[ESPACO_EUCLIDIANO])
NOMES_DO_ESPACO = list(preprocessador.get_feature_names_out())

# Dissimilaridade de Gower sobre o espaço misto: 17 numéricas e ordinais pela
# diferença normalizada, 5 nominais por igualdade. É a segunda geometria que a
# fase 3 deixou preparada, e a única que lê as nominais sem *one-hot*.
GOWER = dist.gower(base, ESPACO_EUCLIDIANO, ESPACO_MISTO)

print(
    f"Espaço euclidiano: {EUCLIDIANO.shape[0]} × {EUCLIDIANO.shape[1]} · "
    f"média {abs(EUCLIDIANO.mean()):.1e}, desvio {EUCLIDIANO.std():.3f}"
)
print(
    f"Matriz de Gower:   {GOWER.shape[0]} × {GOWER.shape[1]} · "
    f"{GOWER.nbytes / 1e6:.1f} MB · faixa [{GOWER.min():.3f}, {GOWER.max():.3f}]"
)

# 1 · Seleção das técnicas de modelagem

*Select modeling technique* — a primeira tarefa da fase 4 do CRISP-DM.

A tentação, aqui, é listar todos os algoritmos de agrupamento que a biblioteca oferece e
rodar todos. O resultado é um comparativo de sete linhas em que a escolha sai por décimo de
silhueta — e nenhuma delas explica por que aquele algoritmo *poderia* estar certo.

A seleção deste bloco funciona ao contrário: cada família de algoritmos carrega uma
**premissa sobre a forma dos grupos**, e a premissa é uma afirmação sobre a carteira que
pode ser medida antes de rodar qualquer coisa. A família entra se a premissa for plausível;
entra **com previsão de falha** se a medida a enfraquecer — e a previsão é conferida no
bloco 3; e fica fora se for incompatível com a geometria ou com uma decisão já tomada.

## O que a técnica precisa entregar

Antes da premissa, a exigência. O Canvas do Problema define o produto desta fase, e ele
restringe o conjunto de técnicas admissíveis mais do que qualquer consideração estatística:

| Exigência do Canvas | Consequência para a escolha da técnica |
|:---|:---|
| Entre **2 e 5** linhas de produto | Técnica com número de grupos controlável, ou com parâmetro calibrável para cair nessa faixa |
| Cada segmento vira **ação comercial** | O grupo precisa ser descritível em atributos de negócio — o que exclui agrupar sobre componentes |
| A carteira **inteira** recebe política | Técnica que deixa metade dos imóveis sem grupo não serve, salvo se o "sem grupo" for entrega (é o caso do ruído do DBSCAN, com teto de 10%) |
| Imóvel novo precisa ser classificado | A técnica precisa ter — ou admitir — regra de atribuição para registro não visto, senão a fase 6 não existe |

O último item costuma ser esquecido e é eliminatório na prática: um agrupamento que só sabe
rotular os 2.350 imóveis que já viu não vira aplicação.

## O diagnóstico que antecede a escolha

Três medidas sobre o espaço, nenhuma delas um algoritmo de agrupamento. Elas dizem que tipo
de estrutura existe para ser encontrada — e, portanto, que tipo de técnica tem chance.

In [ ]:
# Diagnóstico do espaço, antes de qualquer algoritmo. Três medidas, três premissas.
distancias_euclidianas = pdist(EUCLIDIANO)
nao_nulas = distancias_euclidianas[distancias_euclidianas > 0]
gower_pares = GOWER[np.triu_indices_from(GOWER, k=1)]

# Regra usual do DBSCAN: minPts = 2 × dimensão. Fixada aqui porque a curva
# k-distância depende dela, e a curva é diagnóstico deste bloco.
MIN_AMOSTRAS = 2 * len(ESPACO_EUCLIDIANO)
curva_k_distancia = den.curva_k_distancia(EUCLIDIANO, MIN_AMOSTRAS)
posicao_joelho, EPS_JOELHO = den.joelho_k_distancia(curva_k_distancia)

diagnostico = pd.Series(
    {
        "registros por dimensão": round(LINHAS / len(ESPACO_EUCLIDIANO), 1),
        "duplicatas exatas no espaço": int(
            pd.DataFrame(EUCLIDIANO).round(9).duplicated().sum()
        ),
        "distância euclidiana mediana": round(float(np.median(nao_nulas)), 3),
        "contraste p99/p01": round(
            float(np.percentile(nao_nulas, 99) / np.percentile(nao_nulas, 1)), 2
        ),
        "coef. de variação das distâncias": round(float(nao_nulas.std() / nao_nulas.mean()), 3),
        "dissimilaridade de Gower mediana": round(float(np.median(gower_pares)), 3),
        "k-distância mediana": round(float(np.median(curva_k_distancia)), 3),
        "k-distância no joelho": round(EPS_JOELHO, 3),
        "razão joelho / mediana": round(EPS_JOELHO / float(np.median(curva_k_distancia)), 2),
    }
)

fig, eixos = plt.subplots(1, 3, figsize=(16, 4.4), constrained_layout=True)

sns.histplot(nao_nulas, bins=60, color=BARRA, ax=eixos[0])
eixos[0].axvline(float(np.median(nao_nulas)), color=REFERENCIA, ls="dashed", lw=2)
eixos[0].set_xlabel("Distância euclidiana entre pares")
eixos[0].set_ylabel("Pares")
eixos[0].set_title("Uma moda, sem vale — não há ilhas de imóveis")

sns.histplot(gower_pares, bins=60, color=DESTAQUE, ax=eixos[1])
eixos[1].axvline(float(np.median(gower_pares)), color=REFERENCIA, ls="dashed", lw=2)
eixos[1].set_xlabel("Dissimilaridade de Gower entre pares")
eixos[1].set_ylabel("Pares")
eixos[1].set_title("O espaço misto conta a mesma história")

eixos[2].plot(np.arange(len(curva_k_distancia)), curva_k_distancia, color=BARRA, lw=2)
eixos[2].axhline(EPS_JOELHO, color=ALERTA, ls="dashed", lw=2)
eixos[2].axvline(posicao_joelho, color=ALERTA, ls="dotted", lw=1.5)
eixos[2].set_xlabel(f"Imóveis ordenados pela distância ao {MIN_AMOSTRAS}º vizinho")
eixos[2].set_ylabel("Distância")
eixos[2].set_title(f"Curva k-distância — joelho suave em {EPS_JOELHO:.2f}")

fig.suptitle(
    "Diagnóstico do espaço: a carteira é uma nuvem contínua, não um arquipélago",
    fontsize=13,
    y=1.07,
)
salvar(
    fig,
    "diagnostico-do-espaco",
    diagnostico.rename("valor").to_frame().reset_index(names="medida"),
)

diagnostico.to_frame("medida na base de modelagem")

### O que o diagnóstico já diz

Três leituras, e todas restringem a escolha:

* **a nuvem é contínua.** O histograma das distâncias tem uma moda só e nenhum vale, e o de
  Gower repete o desenho. Não existem ilhas de imóveis separadas por regiões vazias — o que
  é o esperado num mercado imobiliário real, onde o produto varia por gradiente de tamanho e
  padrão, não por classes disjuntas. **Consequência:** quem procura região densa cercada de
  vazio (DBSCAN, Mean Shift) vai ter dificuldade, e a silhueta de qualquer partição será
  modesta — porque a silhueta mede exatamente a separação que esta carteira não tem;
* **138 registros por dimensão.** Longe do regime em que a maldição da dimensionalidade
  achata as distâncias: o contraste entre o percentil 99 e o percentil 1 é de **4,28**, e o
  coeficiente de variação das distâncias, **0,30**. Distância euclidiana ainda discrimina
  aqui, e é isso que autoriza a família particional;
* **o joelho da curva k-distância é suave.** A razão entre o joelho e a mediana é **1,29** —
  não há degrau. Um `eps` calibrado no joelho vai varrer quase toda a carteira de uma vez.

A terceira leitura é uma **previsão**, e fica registrada como tal: o DBSCAN entra no
comparativo esperando falhar de um modo específico — um grupo gigante, ou ruído acima do
teto de 10%. O bloco 3 confere se a previsão se cumpre. Prever e conferir vale mais do que
excluir por antecipação, porque a previsão pode estar errada — e, se estiver, é o
diagnóstico que se revisa.

## As cinco técnicas selecionadas

| Família | Premissa sobre a forma dos grupos | Estado da premissa nesta base | Veredicto |
|:---|:---|:---|:---|
| **Particional por centroide** (K-Means) | Grupos convexos, de dispersão parecida, em torno de uma média | Plausível: espaço padronizado, contraste preservado | entra |
| **Particional por medoide** (K-Medoids sobre Gower) | Idem, mas o centro é um registro real e a medida pode ser qualquer dissimilaridade | Plausível, e é a única que lê as 5 nominais sem *one-hot* | entra |
| **Hierárquica aglomerativa** (4 ligações) | Existe hierarquia de fusões; o corte sai da altura do salto | Plausível — e não exige *k* de antemão, o que a torna a segunda opinião sobre o número de grupos | entra |
| **Densidade** (DBSCAN, Mean Shift) | Grupos são regiões densas separadas por regiões esparsas | **Enfraquecida**: nuvem unimodal, joelho suave | entra, com previsão de falha |
| **Probabilística** (GMM) | Os dados vêm de uma mistura de gaussianas; cada registro tem grau de pertencimento | Parcial: entrega o grau de pertencimento, que nenhuma outra dá; a gaussianidade é aproximação | entra, com ressalva |

Nenhuma família foi descartada por premissa violada. A nuvem contínua **enfraquece** a
premissa de densidade, mas não a refuta antes de medir.

In [ ]:
familias = pd.DataFrame(
    [
        {
            "familia": "Particional (centroide)",
            "algoritmo": "K-Means",
            "espaco": f"euclidiano ({len(ESPACO_EUCLIDIANO)})",
            "premissa": "Grupos convexos de dispersão parecida em torno de uma média",
            "medida_que_sustenta": f"contraste p99/p01 = {diagnostico['contraste p99/p01']}",
            "k_controlavel": True,
            "classifica_novo": "sim — centroide mais próximo",
            "veredicto": "entra",
        },
        {
            "familia": "Particional (medoide)",
            "algoritmo": "K-Medoids",
            "espaco": f"misto — Gower ({len(ESPACO_EUCLIDIANO) + len(ESPACO_MISTO)})",
            "premissa": "Idem, com centro em registro real e dissimilaridade arbitrária",
            "medida_que_sustenta": (
                f"Gower mediana = {diagnostico['dissimilaridade de Gower mediana']}"
            ),
            "k_controlavel": True,
            "classifica_novo": "sim — medoide mais próximo",
            "veredicto": "entra",
        },
        {
            "familia": "Hierárquica",
            "algoritmo": "Aglomerativa (4 ligações)",
            "espaco": f"euclidiano ({len(ESPACO_EUCLIDIANO)})",
            "premissa": "Há hierarquia de fusões; a altura do salto indica onde cortar",
            "medida_que_sustenta": "não exige k a priori — segunda opinião sobre o número de grupos",
            "k_controlavel": True,
            "classifica_novo": "não nativamente — exige regra externa",
            "veredicto": "entra",
        },
        {
            "familia": "Densidade",
            "algoritmo": "DBSCAN e Mean Shift",
            "espaco": f"euclidiano ({len(ESPACO_EUCLIDIANO)})",
            "premissa": "Grupos são regiões densas separadas por regiões esparsas",
            "medida_que_sustenta": (
                f"joelho/mediana = {diagnostico['razão joelho / mediana']} — sem degrau"
            ),
            "k_controlavel": False,
            "classifica_novo": "não nativamente (DBSCAN)",
            "veredicto": "entra, com previsão de falha",
        },
        {
            "familia": "Probabilística",
            "algoritmo": "GMM",
            "espaco": f"euclidiano ({len(ESPACO_EUCLIDIANO)})",
            "premissa": "Mistura de gaussianas; cada imóvel tem grau de pertencimento",
            "medida_que_sustenta": "gaussianidade é aproximação — conferida no bloco 3",
            "k_controlavel": True,
            "classifica_novo": "sim — probabilidade a posteriori",
            "veredicto": "entra, com ressalva",
        },
    ]
)
io.salvar_tabela(familias, SECAO, "familias-de-tecnicas", indice=False)

for _, familia in familias.iterrows():
    registrar(
        "1 · Técnicas",
        f"Incluir {familia.algoritmo} no comparativo",
        familia.familia,
        familia.premissa,
        familia.medida_que_sustenta,
        acao="técnica selecionada",
    )

familias[["familia", "algoritmo", "espaco", "k_controlavel", "classifica_novo", "veredicto"]]

## O que fica de fora, e por quê

Quatro exclusões. Nenhuma é sobre desempenho — todas são sobre premissa, sobre decisão já
medida na fase 3, ou sobre impossibilidade aritmética.

**1 · A métrica cosseno e a Manhattan.** As três métricas não são reescalas uma da outra: a
tabela abaixo mostra que distribuem as distâncias entre pares de formas diferentes, e
métrica diferente muda quem é vizinho de quem. A escolha é a euclidiana, e o motivo é de
negócio: a cosseno normaliza o **módulo** do vetor e mede só a direção — dois imóveis com o
mesmo perfil de atributos e o dobro do tamanho ficariam à distância zero. Porte é
exatamente o que a Meridiano precisa separar. A Manhattan é defensável e produziria
partições parecidas; fica fora porque a inércia do K-Means é definida sobre soma de
quadrados, e usar uma métrica enquanto se otimiza outra é incoerência silenciosa.

**2 · O espaço com *one-hot*.** Decidido e medido na fase 3: 27 colunas binárias, 8
categorias abaixo de 1% da carteira, a mais rara em um único imóvel (*z* ≈ 48) e ARI de 0,26
contra o espaço de 17 variáveis. As nominais entram por Gower, que é a família 2.

**3 · Agrupar sobre componentes do PCA.** Também decidido na fase 3, por motivo de negócio:
segmento descrito como "alto em PC1" não vira linha de produto. O PCA aparece neste notebook
**uma vez**, para projetar os grupos em duas dimensões *depois* de formados — o que é
legítimo, porque a projeção não define grupo nenhum.

**4 · Ward sobre Gower, e K-Means sobre Gower.** As duas são impossíveis, não indesejáveis.
O critério de Ward minimiza variância, o que pressupõe espaço euclidiano — aplicá-lo a uma
matriz de dissimilaridade devolve altura sem significado, e `src/model/hierarquica.py`
levanta exceção em vez de devolver o número. O K-Means precisa da média dos pontos de um
grupo, e **não existe média** numa dissimilaridade sem coordenadas: é por isso que o
K-Medoids existe.

A célula abaixo **executa** a exceção em vez de descrevê-la. Um limite demonstrado no
notebook é diferente de um limite afirmado na prosa.

In [ ]:
metricas = dist.comparar_metricas(EUCLIDIANO, ("euclidean", "cityblock", "cosine"))
io.salvar_tabela(metricas.reset_index(), SECAO, "metricas-de-distancia", indice=False)
display(metricas)

# O limite do Ward sobre Gower, demonstrado.
try:
    hier.matriz_ligacao(distancias=GOWER, ligacao="ward")
    LIMITE_WARD_GOWER = "não levantou exceção — conferir src/model/hierarquica.py"
except ValueError as erro:
    LIMITE_WARD_GOWER = str(erro)
print(f"Ward sobre Gower → {LIMITE_WARD_GOWER}")

registrar(
    "1 · Técnicas",
    "Usar distância euclidiana no espaço de 17 variáveis",
    "Escolha da medida de distância",
    "A cosseno normaliza o módulo do vetor e mede só a direção: dois imóveis de mesmo "
    "perfil e tamanhos diferentes ficariam à distância zero, e porte é o que a Meridiano "
    "precisa separar. A Manhattan é coerente, mas a inércia do K-Means é soma de quadrados.",
    f"distância mediana: euclidiana {metricas.loc['euclidean', 'mediana']:.2f}, "
    f"Manhattan {metricas.loc['cityblock', 'mediana']:.2f}, "
    f"cosseno {metricas.loc['cosine', 'mediana']:.2f}",
)
registrar(
    "1 · Técnicas",
    "Não agrupar sobre o espaço com *one-hot*",
    "Codificação das nominais",
    "Medido na fase 3: 8 das 27 categorias abaixo de 1% da carteira e a mais rara em um "
    "único imóvel. As nominais entram por Gower, que não infla a dimensão.",
    "ARI 0,26 contra o espaço de 17 variáveis (fase 3)",
    acao="técnica descartada",
)
registrar(
    "1 · Técnicas",
    "Não agrupar sobre componentes principais",
    "Redução de dimensionalidade",
    "Segmento descrito por componente não vira linha de produto. O PCA fica só como "
    "projeção 2D, aplicada depois do agrupamento.",
    f"{manifesto_preparacao['componentes_para_80pct']} componentes para 80% da "
    "variância (fase 3)",
    acao="técnica descartada",
)
registrar(
    "1 · Técnicas",
    "Não aplicar Ward nem K-Means sobre a matriz de Gower",
    "Compatibilidade entre algoritmo e geometria",
    "Ward minimiza variância e K-Means calcula médias: as duas operações pressupõem "
    "coordenadas euclidianas, que uma matriz de dissimilaridade não tem. Sobre Gower, o "
    "análogo correto é o K-Medoids.",
    f"exceção do projeto: {LIMITE_WARD_GOWER[:58]}…",
    acao="técnica descartada",
)

# 2 · Desenho do experimento

*Generate test design* — a segunda tarefa da fase 4, e a que o aprendizado não supervisionado
torna difícil.

Num problema supervisionado o desenho é conhecido: separa-se um conjunto de teste, treina-se
no resto, mede-se o erro contra o rótulo. Aqui **não há rótulo**. Não existe a resposta certa
contra a qual medir, e a métrica interna — silhueta, Davies-Bouldin — mede geometria, não
acerto: ela diz se os grupos estão separados, nunca se estão certos.

A pergunta que o desenho precisa responder, então, é outra:

> **O que este resultado tem de reprodutível?** Se a partição muda quando se tira um quinto
> dos imóveis, quando se troca a semente ou quando chega um imóvel novo, ela é um artefato do
> conjunto — e a diretoria estaria alocando orçamento sobre ruído.

Três provas independentes respondem a isso, e as três estão desenhadas antes de qualquer
algoritmo rodar.

| Prova | Pergunta | Como se mede | Reprova quando |
|:---|:---|:---|:---|
| **1 · Varredura** | Quantos grupos, e com que geometria? | Métricas internas para cada *k* e cada configuração | A configuração não cai na faixa de negócio |
| **2 · Reamostragem** | A partição sobrevive a perder 20% da carteira? | 20 subamostras de 80%, reagrupadas do zero, ARI contra a referência | ARI abaixo do piso do Canvas |
| **3 · *Hold-out*** | O modelo sabe classificar imóvel que nunca viu? | Ajuste em 80%, atribuição dos 20% restantes, silhueta e ARI no conjunto retido | Silhueta despenca no retido, ou a atribuição discorda do modelo completo |

A prova 3 é o análogo mais próximo do *hold-out* supervisionado, e vale a pena dizer o que
ela mede e o que não mede: ela **não** verifica se os grupos estão certos — nada verifica
isso sem rótulo. Ela verifica se a estrutura encontrada nos 80% descreve os 20% que ficaram
de fora, que é a única forma de saber, antes da fase 6, se o modelo serve para imóvel novo.

## Vedação do rótulo, e o que ela custa

O preço e o bairro não são abertos neste notebook. A consequência prática precisa ficar
clara: **um dos critérios do Canvas não pode ser medido aqui.**

> "Independência do rótulo reservado: ARI contra quartis de preço < 0,40."

Esse critério exige `preco_venda`, e medi-lo agora significaria olhar o preço antes de
escolher o modelo — exatamente o que a fase 3 impediu por construção. Ele fica declarado
aqui como **critério pendente**, a ser verificado na fase 5 sobre o modelo já escolhido. Se
reprovar lá, a decisão volta para este notebook, e o registro mostra por quê.

In [ ]:
# Os critérios de aceitação, declarados ANTES de qualquer algoritmo rodar.
# Todos vêm do Canvas do Problema (docs/pbl/canvas-do-problema.md, seção 6) e de
# `src/config.py` — nenhum foi inventado aqui.
SILHUETA_MINIMA = config.SILHUETA_MINIMA
K_MINIMO, K_MAXIMO = config.K_MINIMO_NEGOCIO, config.K_MAXIMO_NEGOCIO
RUIDO_MAXIMO = config.RUIDO_MAXIMO
ARI_MINIMO = 0.75
MENOR_SEGMENTO_MINIMO = 0.02

criterios = pd.DataFrame(
    [
        {
            "criterio": "Número de grupos",
            "valor": f"entre {K_MINIMO} e {K_MAXIMO}",
            "origem": "Canvas · orçamento financia até 5 ações diferenciadas",
            "medido_em": "prova 1 — varredura",
        },
        {
            "criterio": "Silhueta",
            "valor": f"≥ {SILHUETA_MINIMA:.2f}",
            "origem": "Canvas · piso abaixo do qual não há separação defensável",
            "medido_em": "prova 1 — varredura",
        },
        {
            "criterio": "Menor segmento",
            "valor": f"≥ {100 * MENOR_SEGMENTO_MINIMO:.0f}% da carteira",
            "origem": "Canvas · abaixo disso não há massa para ação própria",
            "medido_em": "prova 1 — varredura",
        },
        {
            "criterio": "Ruído (métodos de densidade)",
            "valor": f"≤ {100 * RUIDO_MAXIMO:.0f}% da carteira",
            "origem": "Canvas · imóvel sem grupo é entrega, mas não pode ser a regra",
            "medido_em": "prova 1 — varredura",
        },
        {
            "criterio": "Estabilidade sob reamostragem de 80%",
            "valor": f"ARI ≥ {ARI_MINIMO:.2f}",
            "origem": "Canvas · revisão 2 do quadro de critérios",
            "medido_em": "prova 2 — reamostragem",
        },
        {
            "criterio": "Generalização ao conjunto retido",
            "valor": f"ARI ≥ {ARI_MINIMO:.2f} e silhueta ≥ {SILHUETA_MINIMA:.2f}",
            "origem": "Desenho deste notebook · exigência da fase 6",
            "medido_em": "prova 3 — hold-out",
        },
        {
            "criterio": "Independência do rótulo reservado",
            "valor": "ARI contra quartis de preço < 0,40",
            "origem": "Canvas · revisão 3 do quadro de critérios",
            "medido_em": "PENDENTE — fase 5, com o rótulo aberto",
        },
    ]
)
io.salvar_tabela(criterios, SECAO, "criterios-de-aceitacao", indice=False)

for _, criterio in criterios.iterrows():
    registrar(
        "2 · Desenho",
        f"{criterio.criterio}: {criterio.valor}",
        "Critério de aceitação",
        criterio.origem,
        f"medido na {criterio.medido_em}",
        acao="critério declarado",
    )

criterios

## As duas regras que o bloco 4 vai obedecer

Critério não basta: falta dizer **o que fazer quando mais de um candidato passa**. Sem essa
regra escrita antes, a escolha final vira preferência com aparência de método.

### Regra 1 · Como se escolhe *k*

> **O maior *k* que passa em todos os critérios, dentro do teto de cinco.**

O sentido é de negócio, não estatístico: mais linhas de produto significam política mais
granular, e granularidade é o que a Meridiano ganha com o projeto — desde que cada linha se
sustente sozinha. O maior *k* que ainda passa é o ponto em que a granularidade acaba. Note
que a regra **não** é "o *k* de maior silhueta": a silhueta cresce quando *k* diminui, e
levada ao limite ela devolve dois grupos, "caro" e "barato", que a diretoria já tem.

### Regra 2 · Como se desempata entre algoritmos

Na ordem, e só se passa para a seguinte em caso de empate:

1. **passar em todos os critérios** — quem reprova em um, sai, sem compensação por outro;
2. **estabilidade no pior caso** — o menor ARI entre as repetições, não a média. Média alta
   com mínimo baixo significa que existe uma reamostra em que a partição se desfez, e é
   exatamente essa reamostra que a próxima carga de dados pode ser. É a mesma lógica que o
   projeto já aplica à silhueta por grupo: a média esconde o grupo esgarçado;
3. **silhueta**, dentro do mesmo espaço;
4. **equilíbrio e simplicidade** — grupos de tamanho operável e um modelo que a diretoria
   consegue entender sem um curso.

### Uma regra de comparação, para não somar o que não se soma

Silhueta calculada no espaço euclidiano e silhueta calculada sobre Gower **não são o mesmo
número**: são medidas em geometrias diferentes, e ordenar as duas numa coluna só é erro. O
modelo de Gower é avaliado contra os critérios **dentro do seu próprio espaço**, e a
comparação entre os dois espaços é feita por ARI — que só olha concordância de partição — e
pela leitura de negócio do resultado.

In [ ]:
# Parâmetros do desenho, fixados aqui e usados nos blocos 3 e 4.
K_VARREDURA = range(2, 11)  # a varredura vai além do teto de negócio de propósito:
                            # é preciso ver o comportamento fora da faixa para saber
                            # se o teto está cortando estrutura real ou ruído.
REPETICOES = 20             # reamostras por configuração na prova 2
FRACAO = 0.80               # tamanho de cada reamostra e do conjunto de ajuste
SEMENTE = config.SEMENTE

# Prova 3 — a partição hold-out. Sorteio simples: sem rótulo, não há o que
# estratificar, e estratificar por uma variável do espaço enviesaria o teste
# a favor dela.
gerador = np.random.default_rng(SEMENTE)
embaralhado = gerador.permutation(LINHAS)
corte = int(FRACAO * LINHAS)
AJUSTE = np.sort(embaralhado[:corte])
RETIDO = np.sort(embaralhado[corte:])

assert len(set(AJUSTE) & set(RETIDO)) == 0, "ajuste e hold-out não podem se cruzar"
assert len(AJUSTE) + len(RETIDO) == LINHAS, "a partição hold-out perdeu registros"


def preparar(indices_ajuste, indices_aplicar=None):
    """Ajusta a receita **dentro** da amostra e a aplica onde for pedido.

    É aqui que a decisão da fase 3 — gravar a receita, não o dado padronizado —
    paga o que prometeu. Cada reamostra e o conjunto de ajuste calculam as
    próprias médias e desvios; o conjunto retido recebe os do ajuste, nunca os
    seus. Padronizar tudo de uma vez antes de reamostrar devolveria estabilidade
    otimista, porque cada reamostra já teria visto a carteira inteira pela média.
    """
    receita = clone(preprocessador)
    matriz = receita.fit_transform(base.iloc[indices_ajuste][ESPACO_EUCLIDIANO])
    if indices_aplicar is None:
        return matriz
    return matriz, receita.transform(base.iloc[indices_aplicar][ESPACO_EUCLIDIANO])


def rotular_kmeans(dados, k, semente=SEMENTE):
    """K-Means com 20 reinicializações — o padrão do projeto."""
    return KMeans(n_clusters=k, init="k-means++", n_init=20, random_state=semente).fit_predict(dados)


def rotular_ward(dados, k, semente=SEMENTE):
    """Aglomerativa de Ward. Determinística: a semente não a afeta, e por isso a
    prova 2 mede nela só o efeito da reamostragem — que é o que interessa."""
    return AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(dados)


def rotular_gmm(covariancia):
    """Devolve o rotulador de GMM para um tipo de covariância."""

    def rotular(dados, k, semente=SEMENTE):
        return GaussianMixture(
            n_components=k,
            covariance_type=covariancia,
            n_init=5,
            max_iter=500,
            random_state=semente,
        ).fit_predict(dados)

    return rotular


def estabilidade(rotular, k, repeticoes=REPETICOES, fracao=FRACAO, semente=SEMENTE):
    """Prova 2, genérica: reamostra, reajusta a receita, reagrupa e compara.

    `src.model.estabilidade.estabilidade_bootstrap` faz o mesmo, mas só para
    K-Means e sobre uma matriz já padronizada. Aqui o rotulador é parâmetro (para
    comparar famílias) e a padronização acontece dentro da reamostra (para não
    vazar). A generalização volta para o código como pendência, no fim.
    """
    aleatorio = np.random.default_rng(semente)
    referencia = rotular(EUCLIDIANO, k, semente)
    tamanho = int(fracao * LINHAS)
    aris = []
    for _ in range(repeticoes):
        indices = np.sort(aleatorio.choice(LINHAS, size=tamanho, replace=False))
        rotulos = rotular(preparar(indices), k, int(aleatorio.integers(1e6)))
        aris.append(float(adjusted_rand_score(referencia[indices], rotulos)))
    return np.array(aris)


print(f"Conjunto de ajuste: {pt(len(AJUSTE))} imóveis · retido: {pt(len(RETIDO))}")
print(f"Prova 2: {REPETICOES} reamostras de {int(FRACAO * LINHAS)} imóveis por configuração")

In [ ]:
# O esquema do desenho, desenhado. Três linhas: a carteira, a prova 3 e a prova 2.
fig, ax = plt.subplots(figsize=(13, 4.6))

ax.barh(2, LINHAS, color=NEUTRO, height=0.5)
ax.text(LINHAS / 2, 2, f"{pt(LINHAS)} imóveis · base de modelagem", ha="center", va="center",
        color="white", fontweight="bold", fontsize=10)

ax.barh(1, len(AJUSTE), color=BARRA, height=0.5)
ax.barh(1, len(RETIDO), left=len(AJUSTE), color=ALERTA, height=0.5)
ax.text(len(AJUSTE) / 2, 1, f"ajuste · {pt(len(AJUSTE))}", ha="center", va="center",
        color="white", fontweight="bold", fontsize=9)
ax.text(len(AJUSTE) + len(RETIDO) / 2, 1, f"retido · {pt(len(RETIDO))}", ha="center",
        va="center", color="white", fontweight="bold", fontsize=8)

deslocamentos = np.linspace(0, LINHAS - int(FRACAO * LINHAS), 6).astype(int)
for posicao, deslocamento in enumerate(deslocamentos):
    ax.barh(0.15 - 0.06 * posicao, int(FRACAO * LINHAS), left=deslocamento, color=DESTAQUE,
            height=0.045, alpha=0.9)
ax.text(LINHAS + 40, -0.05, f"× {REPETICOES} reamostras\nde {int(100 * FRACAO)}%",
        va="center", fontsize=9, color=config.AZUL_PROFUNDO)

ax.set_yticks([2, 1, 0])
ax.set_yticklabels(["Carteira", "Prova 3 · hold-out", "Prova 2 · reamostragem"])
ax.set_xlabel("Imóveis")
ax.set_xlim(0, LINHAS * 1.18)
ax.set_ylim(-0.45, 2.5)
ax.grid(False)
ax.set_title("Desenho do experimento — a receita é reajustada dentro de cada bloco azul")

salvar(
    fig,
    "desenho-do-experimento",
    pd.DataFrame(
        [
            {"bloco": "carteira", "imoveis": LINHAS, "papel": "base de modelagem"},
            {"bloco": "ajuste", "imoveis": len(AJUSTE), "papel": "prova 3 — onde a receita é ajustada"},
            {"bloco": "retido", "imoveis": len(RETIDO), "papel": "prova 3 — nunca visto pelo modelo"},
            {"bloco": "reamostra", "imoveis": int(FRACAO * LINHAS), "papel": f"prova 2 — {REPETICOES} sorteios sem reposição"},
        ]
    ),
)

# A grade: toda configuração que será construída no bloco 3.
grade = pd.DataFrame(
    [
        {"familia": "Particional", "configuracao": "K-Means", "espaco": "euclidiano",
         "parametro_varrido": f"k = {K_VARREDURA.start}…{K_VARREDURA.stop - 1}",
         "execucoes": len(K_VARREDURA)},
        {"familia": "Particional", "configuracao": "K-Medoids", "espaco": "Gower",
         "parametro_varrido": "no k decidido em 3.3", "execucoes": 1},
        {"familia": "Hierárquica", "configuracao": "Aglomerativa", "espaco": "euclidiano",
         "parametro_varrido": f"{len(hier.LIGACOES)} ligações", "execucoes": len(hier.LIGACOES)},
        {"familia": "Densidade", "configuracao": "DBSCAN", "espaco": "euclidiano",
         "parametro_varrido": "13 valores de eps, minPts = 2d", "execucoes": 13},
        {"familia": "Densidade", "configuracao": "Mean Shift", "espaco": "euclidiano",
         "parametro_varrido": "4 quantis de largura de banda", "execucoes": 4},
        {"familia": "Probabilística", "configuracao": "GMM", "espaco": "euclidiano",
         "parametro_varrido": f"4 covariâncias × k = {K_MINIMO}…{K_MAXIMO}",
         "execucoes": 4 * (K_MAXIMO - K_MINIMO + 1)},
    ]
)
grade.loc[len(grade)] = {
    "familia": "—", "configuracao": "TOTAL", "espaco": "—",
    "parametro_varrido": "—", "execucoes": int(grade.execucoes.sum()),
}
io.salvar_tabela(grade, SECAO, "grade-do-experimento", indice=False)

registrar(
    "2 · Desenho",
    "Reajustar o pré-processador dentro de cada reamostra e do conjunto de ajuste",
    "Protocolo sem vazamento",
    "Padronizar a carteira inteira antes de reamostrar faria cada reamostra herdar médias "
    "que já viram os registros tratados como desconhecidos: estabilidade otimista no "
    "notebook e comportamento pior em produção.",
    f"{REPETICOES} reamostras × {int(FRACAO * LINHAS)} imóveis, receita reajustada em cada",
    acao="critério declarado",
)
registrar(
    "2 · Desenho",
    f"Escolher o maior k que passa em todos os critérios, com teto em {K_MAXIMO}",
    "Regra de escolha de k",
    "Mais linhas de produto significam política mais granular, desde que cada uma se "
    "sustente. A regra não é a maior silhueta: a silhueta cresce quando k cai e levaria a "
    "dois grupos, 'caro' e 'barato', que a diretoria já tem.",
    f"faixa avaliada: k = {K_VARREDURA.start}…{K_VARREDURA.stop - 1}",
    acao="critério declarado",
)
registrar(
    "2 · Desenho",
    "Desempatar por estabilidade no pior caso, não pela média",
    "Regra de desempate entre algoritmos",
    "Média alta com mínimo baixo significa que existe uma reamostra em que a partição se "
    "desfez — e é essa reamostra que a próxima carga de dados pode ser.",
    f"{int(grade.execucoes.iloc[-1])} execuções na grade do experimento",
    acao="critério declarado",
)

grade

# 3 · Construção dos modelos

*Build model* — a terceira tarefa. Com as técnicas escolhidas e o desenho fixado, o que resta
é executar. Este bloco constrói as 47 configurações da grade, e cada subseção termina em um
parâmetro fixado, com o número que o fixou.

A ordem é deliberada: **primeiro *k*, depois os algoritmos**. O número de grupos é a decisão
que a diretoria vai enxergar; os algoritmos disputam entre si *depois*, sobre o mesmo *k*,
para que a comparação do bloco 4 não confunda "algoritmo melhor" com "número de grupos
diferente".

## 3.1 · A varredura de *k*

Quatro métricas, e uma leitura para cada uma:

* **inércia** — soma das distâncias ao quadrado até o centroide. Só cai quando *k* sobe;
  sozinha, não escolhe nada. O que se lê é o **joelho**: o ponto onde acrescentar um grupo
  para de comprar redução. `src/model/particional.py` implementa o critério da maior
  distância à corda — que é o que se faz a olho, escrito de forma reprodutível;
* **silhueta** — separação relativa: quão mais perto um imóvel está do próprio grupo do que
  do vizinho mais próximo. É a métrica que o Canvas usa como piso;
* **Davies-Bouldin** — razão entre dispersão interna e separação entre centros. Menor é
  melhor;
* **Calinski-Harabasz** — variância entre grupos sobre variância dentro. Maior é melhor.

A varredura vai até *k* = 10, além do teto de negócio, de propósito. Sem olhar fora da faixa
não é possível saber se o teto está cortando estrutura real ou apenas ruído.

In [ ]:
varredura = part.varrer_k(EUCLIDIANO, K_VARREDURA.start, K_VARREDURA.stop - 1, semente=SEMENTE)
K_COTOVELO = part.cotovelo(varredura["inercia"])

fig, eixos = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)

eixos[0, 0].plot(varredura.index, varredura.inercia, marker="o", color=BARRA, lw=2)
eixos[0, 0].axvline(K_COTOVELO, color=ALERTA, ls="dashed", lw=2)
eixos[0, 0].set_title(f"Inércia — joelho da corda em k = {K_COTOVELO}")
eixos[0, 0].set_xlabel("k")
eixos[0, 0].set_ylabel("Inércia")

cores = [
    APROVADO if valor >= SILHUETA_MINIMA else ALERTA for valor in varredura.silhueta
]
eixos[0, 1].bar(varredura.index, varredura.silhueta, color=cores)
eixos[0, 1].axhline(SILHUETA_MINIMA, color=REFERENCIA, ls="dashed", lw=2)
eixos[0, 1].bar_label(eixos[0, 1].containers[0], fmt="%.3f", padding=2, fontsize=8)
eixos[0, 1].set_title(f"Silhueta — piso do Canvas em {SILHUETA_MINIMA:.2f}")
eixos[0, 1].set_xlabel("k")
eixos[0, 1].margins(y=0.18)

eixos[1, 0].plot(varredura.index, varredura.davies_bouldin, marker="o", color=BARRA, lw=2)
eixos[1, 0].set_title("Davies-Bouldin — menor é melhor")
eixos[1, 0].set_xlabel("k")

eixos[1, 1].plot(varredura.index, varredura.calinski_harabasz, marker="o", color=BARRA, lw=2)
eixos[1, 1].set_title("Calinski-Harabasz — maior é melhor")
eixos[1, 1].set_xlabel("k")

for eixo in eixos.flat:
    eixo.set_xticks(list(K_VARREDURA))
    eixo.axvspan(K_MINIMO - 0.4, K_MAXIMO + 0.4, color=REFERENCIA, alpha=0.10)

fig.suptitle(
    f"Varredura de k — a faixa sombreada é o teto de negócio ({K_MINIMO} a {K_MAXIMO})",
    fontsize=13,
    y=1.04,
)
salvar(fig, "varredura-de-k", varredura.reset_index())

print(f"Joelho da inércia: k = {K_COTOVELO}")
print(f"Maior silhueta:    k = {varredura.silhueta.idxmax()} ({varredura.silhueta.max():.4f})")
print(f"Menor Davies-Bouldin: k = {varredura.davies_bouldin.idxmin()}")
varredura.round(4)

### O que a varredura sozinha diria — e por que não basta

As quatro métricas apontam para lugares diferentes, e isso é o normal, não um problema do
conjunto: cada uma mede uma coisa. A silhueta, o Davies-Bouldin e o Calinski-Harabasz
apontam para **k = 2**; o joelho da inércia aponta para **k = 5**.

Se a decisão saísse daqui, sairia *k* = 2 — e *k* = 2, nesta carteira, é a partição entre
imóvel pequeno e antigo e imóvel grande e novo. É verdadeira, é bem separada e é inútil: a
diretoria não precisa de um algoritmo para saber que a carteira tem imóvel bom e imóvel
ruim, e a política única que ela quer substituir já opera exatamente com essa distinção.

É por isso que a regra declarada no bloco 2 não é "maior silhueta", e sim **o maior *k* que
passa em todos os critérios**. Falta ainda um critério para aplicá-la: a estabilidade.

## 3.2 · A prova 2 — o que sobrevive a perder 20% da carteira

Cada *k* candidato é reamostrado 20 vezes. Em cada reamostra, a receita da fase 3 é
reajustada dentro dela, o K-Means roda do zero com outra semente, e o resultado é comparado
com a partição de referência **restrita aos mesmos imóveis**, pelo índice de Rand ajustado.

O que este teste pega é o *k* que parece bom na métrica e não se reproduz — a armadilha mais
comum da segmentação. Um agrupamento com silhueta razoável e ARI de 0,5 é um agrupamento que
seria outro se a carteira tivesse chegado com outros 470 imóveis.

In [ ]:
linhas_estabilidade = []
for k in K_VARREDURA:
    aris = estabilidade(rotular_kmeans, k)
    linhas_estabilidade.append(
        {
            "k": k,
            "ari_medio": round(float(aris.mean()), 4),
            "ari_minimo": round(float(aris.min()), 4),
            "ari_desvio": round(float(aris.std()), 4),
            "repeticoes_abaixo_do_piso": int((aris < ARI_MINIMO).sum()),
        }
    )
estabilidade_por_k = pd.DataFrame(linhas_estabilidade).set_index("k")

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(estabilidade_por_k.index, estabilidade_por_k.ari_medio, marker="o", lw=2,
        color=BARRA, label="ARI médio")
ax.plot(estabilidade_por_k.index, estabilidade_por_k.ari_minimo, marker="s", lw=2,
        ls="dashed", color=ALERTA, label="ARI mínimo (pior reamostra)")
ax.fill_between(estabilidade_por_k.index, estabilidade_por_k.ari_minimo,
                estabilidade_por_k.ari_medio, color=BARRA, alpha=0.12)
ax.axhline(ARI_MINIMO, color=REFERENCIA, ls="dotted", lw=2)
ax.text(K_VARREDURA.stop - 1, ARI_MINIMO, f" piso {ARI_MINIMO:.2f}", va="bottom", ha="right",
        color=config.AZUL_PROFUNDO, fontsize=9)
ax.set_xticks(list(K_VARREDURA))
ax.set_xlabel("k")
ax.set_ylabel("Índice de Rand ajustado")
ax.set_title(f"Estabilidade sob {REPETICOES} reamostras de {int(100 * FRACAO)}% da carteira")
ax.legend()

salvar(fig, "estabilidade-por-k", estabilidade_por_k.reset_index())
estabilidade_por_k

## 3.3 · A decisão de *k*

A regra do bloco 2 aplicada, critério a critério, sem exceção aberta para nenhum *k*. A
tabela abaixo é a regra rodando: cada linha é um critério, cada coluna um *k*, e o veredicto
sai da conjunção.

O resultado tem uma leitura que merece ser dita antes de aparecer: **k = 4 reprova por
silhueta**, com 0,1388 contra o piso de 0,15. É pouco abaixo — e "pouco abaixo" é exatamente
onde o critério é testado. Mover o piso para 0,13 para acomodar quatro segmentos seria
inverter a ordem entre regra e resultado, que é o que este notebook declarou não fazer no
primeiro parágrafo.

Vale o registro de que **a documentação publicada do projeto afirma *k* = 4**, com silhueta
0,1775. Aquele número foi medido sobre a base antiga: 2.925 imóveis e 21 variáveis, com as
quatro binárias que a fase 3 mediu e removeu. A carteira aqui tem 2.350 imóveis depurados e
17 variáveis. **Não é o mesmo experimento**, e o valor que sai dele não tem por que ser o
mesmo. A divergência fica registrada no fim do notebook, com o que precisa ser revisto.

In [ ]:
gate = pd.DataFrame(index=list(K_VARREDURA))
gate["silhueta"] = varredura.silhueta
gate["menor_grupo_pct"] = (100 * varredura.menor_grupo / LINHAS).round(1)
gate["ari_medio"] = estabilidade_por_k.ari_medio
gate["ari_minimo"] = estabilidade_por_k.ari_minimo

testes = pd.DataFrame(
    {
        f"k dentro de [{K_MINIMO}, {K_MAXIMO}]": [K_MINIMO <= k <= K_MAXIMO for k in gate.index],
        f"silhueta ≥ {SILHUETA_MINIMA:.2f}": gate.silhueta >= SILHUETA_MINIMA,
        f"menor grupo ≥ {100 * MENOR_SEGMENTO_MINIMO:.0f}%": (
            gate.menor_grupo_pct >= 100 * MENOR_SEGMENTO_MINIMO
        ),
        f"ARI médio ≥ {ARI_MINIMO:.2f}": gate.ari_medio >= ARI_MINIMO,
        f"ARI mínimo ≥ {ARI_MINIMO:.2f}": gate.ari_minimo >= ARI_MINIMO,
    },
    index=gate.index,
)
gate["passa_em_tudo"] = testes.all(axis=1)

# A regra declarada no bloco 2: o maior k que passa em todos os critérios.
K = int(gate.index[gate.passa_em_tudo].max())

fig, ax = plt.subplots(figsize=(13, 3.6))
sns.heatmap(
    testes.T.astype(int),
    annot=testes.T.map(lambda passou: "✓" if passou else "✗").to_numpy(),
    fmt="",
    cmap=plt.matplotlib.colors.ListedColormap([ALERTA, APROVADO]),
    vmin=0,
    vmax=1,
    cbar=False,
    linewidths=1.2,
    linecolor="white",
    annot_kws={"fontsize": 12, "color": "white", "fontweight": "bold"},
    ax=ax,
)
ax.set_xlabel("k")
ax.set_title(f"Os critérios do bloco 2 aplicados a cada k — o maior que passa em tudo é k = {K}")
ax.tick_params(axis="y", rotation=0)
salvar(fig, "decisao-de-k", gate.join(testes).reset_index(names="k"))

registrar(
    "3 · Construção",
    f"Fixar k = {K}",
    "Regra do maior k que passa em todos os critérios",
    "k = 2 é a partição entre imóvel bom e imóvel ruim, que a política única já opera; "
    f"k = 4 reprova na silhueta ({varredura.silhueta.get(4, float('nan')):.4f} contra o piso "
    f"de {SILHUETA_MINIMA:.2f}) e mover o piso para acomodá-lo inverteria regra e resultado.",
    f"silhueta {gate.loc[K, 'silhueta']:.4f} · ARI médio {gate.loc[K, 'ari_medio']:.4f} · "
    f"ARI mínimo {gate.loc[K, 'ari_minimo']:.4f} · menor grupo {gate.loc[K, 'menor_grupo_pct']}%",
    acao="parâmetro fixado",
)

display(gate.round(4))
print(f"k escolhido: {K}")

## 3.4 · A hierárquica — qual ligação, e por que não a do melhor coeficiente

Quatro critérios de ligação, e a comparação não é preciosismo: eles produzem árvores
diferentes sobre os mesmos dados. `single` liga pelo par mais próximo e produz
**encadeamento** — um grupo que engole a carteira enquanto os outros ficam com um punhado de
imóveis; `complete` liga pelo par mais distante e favorece grupos compactos; `average` fica
no meio; `ward` minimiza a variância interna, e é o único que otimiza o mesmo objetivo do
K-Means.

A métrica própria da hierárquica é o **coeficiente cofenético**: quanto a árvore preserva as
distâncias originais. E aqui ele diz uma coisa que não se pode obedecer.

In [ ]:
ligacoes = hier.comparar_ligacoes(EUCLIDIANO, distancias_euclidianas, K)

LIGACAO = "ward"
matriz_ligacao = hier.matriz_ligacao(EUCLIDIANO, LIGACAO)
alturas = hier.alturas_de_fusao(matriz_ligacao, ultimas=10)
io.salvar_tabela(alturas.reset_index(), SECAO, "alturas-de-fusao", indice=False)

fig, eixos = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)

cores = [APROVADO if nome == LIGACAO else NEUTRO for nome in ligacoes.index]
eixos[0].barh(ligacoes.index, ligacoes.razao_maior_menor, color=cores, log=True)
eixos[0].bar_label(
    eixos[0].containers[0],
    labels=[
        f"{razao:,.0f}×   cofenético {cof:.3f}".replace(",", ".")
        for razao, cof in zip(ligacoes.razao_maior_menor, ligacoes.cofenetico)
    ],
    padding=3,
    fontsize=9,
)
eixos[0].invert_yaxis()
eixos[0].set_xlabel("Razão entre o maior e o menor grupo (escala log)")
eixos[0].set_title("O melhor cofenético produz um grupo de um imóvel só")
eixos[0].margins(x=0.55)

hier.desenhar_dendrograma(
    eixos[1], matriz_ligacao, K, f"{LIGACAO} — dendrograma truncado", ALERTA
)

salvar(fig, "ligacoes-e-dendrograma", ligacoes.reset_index())

registrar(
    "3 · Construção",
    f"Usar a ligação `{LIGACAO}` na hierárquica",
    "Comparação entre os quatro critérios de ligação",
    "`average` tem o melhor coeficiente cofenético e é inutilizável: produz um grupo com "
    "quase toda a carteira e outro com um único imóvel — o efeito de encadeamento. Nenhum "
    "orçamento se reparte assim. `ward` tem o pior cofenético e o único equilíbrio operável.",
    f"razão maior/menor grupo: {LIGACAO} {ligacoes.loc[LIGACAO, 'razao_maior_menor']:.1f}×, "
    f"average {pt(int(ligacoes.loc['average', 'razao_maior_menor']))}×",
    acao="parâmetro fixado",
)

display(ligacoes)
alturas

### A altura que confirma o corte

A tabela de alturas mede o **salto** entre fusões sucessivas. Fundir dois grupos que estavam
distantes é o sinal de que já se passou do ponto: o salto da última fusão (de 2 para 1
grupo) é de 73,6, contra 11,8 na fusão anterior e 2,2 na de antes. A árvore concorda que a
estrutura grossa da carteira está entre dois e quatro grupos — que é a faixa onde a
varredura e a estabilidade já haviam parado.

## 3.5 · O K-Medoids sobre Gower — a segunda geometria

Este é o único modelo que lê as cinco nominais. O centro de cada grupo não é uma média, é um
**imóvel real da carteira** — o que muda a conversa numa reunião: "o representante deste
segmento é o imóvel `pid` tal", em vez de "o centroide tem 2,7 dormitórios".

A pergunta que este modelo responde é se trazer zoneamento, tipo de construção, estilo, tipo
de garagem e ar central para dentro da distância muda a segmentação. E a resposta, medida
abaixo, é que muda — e não na direção que interessa.

In [ ]:
resultado_gower = part.kmedoids(GOWER, K, reinicios=12, semente=SEMENTE)
rotulos_gower = pers.ordenar_canonicamente(base, resultado_gower.rotulos)
silhueta_gower = part.metricas_internas(None, rotulos_gower, GOWER)["silhueta"]

composicao_estilo = perf.composicao_nominal(base, rotulos_gower, "estilo")

fig, ax = plt.subplots(figsize=(12, 4.6))
fundo = np.zeros(len(composicao_estilo))
paleta = [config.AZUL_PROFUNDO, ALERTA, APROVADO, REFERENCIA, NEUTRO]
for cor, coluna in zip(paleta, composicao_estilo.columns):
    ax.barh(composicao_estilo.index.astype(str), composicao_estilo[coluna], left=fundo,
            color=cor, label=coluna)
    fundo = fundo + composicao_estilo[coluna].to_numpy()
ax.set_xlabel("% do segmento")
ax.set_ylabel("Segmento (K-Medoids sobre Gower)")
ax.set_title("A partição de Gower separa a carteira por estilo construtivo")
ax.legend(title="estilo", ncols=5, loc="lower center", bbox_to_anchor=(0.5, -0.38))
salvar(fig, "gower-por-estilo", composicao_estilo.reset_index())

DOMINIO_ESTILO = float(composicao_estilo.drop(columns="outras").max(axis=1).max())

print(f"Custo: {resultado_gower.custo:.1f} · iterações: {resultado_gower.iteracoes}")
print(f"Tamanhos: {np.bincount(rotulos_gower)}")
print(f"Silhueta no espaço de Gower: {silhueta_gower:.4f}")
print(f"Maior concentração de um estilo dentro de um segmento: {DOMINIO_ESTILO:.1f}%")
composicao_estilo

### O que o Gower encontrou, e por que não serve

A silhueta do modelo de Gower é **0,2232**, maior que a do K-Means. E ela não pode ser
comparada com a do K-Means — está medida em outra geometria, e a regra do bloco 2 é explícita
sobre isso. O que decide é a leitura do resultado:

Com peso igual por variável, cada nominal pesa tanto quanto cada medida de área ou nota de
acabamento. Cinco variáveis nominais contra dezessete numéricas é peso demais para
categoria, e o que sai é uma partição comandada por `estilo`: um segmento com **78,3% de
casas térreas**, outro com 43,3% de casas de um pavimento e meio.

Isso é um **fato construtivo**, não uma linha de produto. A Meridiano não tem política
comercial diferente para casa térrea e sobrado — tem para imóvel compacto e antigo *versus*
imóvel amplo e novo, que é o que a diretoria pediu para descobrir. A hipótese "as nominais
melhoram a segmentação" foi testada e derrubada, e fica registrada como tal.

In [ ]:
registrar(
    "3 · Construção",
    "Construir o K-Medoids sobre Gower com k = " + str(K),
    "Particional por medoide, espaço misto",
    "Traz as 5 nominais para dentro da distância sem *one-hot* e devolve um imóvel real "
    "como centro de cada segmento.",
    f"silhueta {silhueta_gower:.4f} no espaço de Gower · tamanhos "
    f"{np.bincount(rotulos_gower).tolist()}",
    acao="parâmetro fixado",
)

md_estilo = composicao_estilo.drop(columns="outras")
segmento_dominado = int(md_estilo.max(axis=1).idxmax())
estilo_dominante = md_estilo.loc[segmento_dominado].idxmax()

registrar(
    "3 · Construção",
    "Não adotar a partição de Gower como segmentação da carteira",
    "Leitura de negócio da partição",
    "Com peso igual por variável, 5 nominais contra 17 numéricas fazem `estilo` comandar a "
    "partição. Casa térrea e sobrado são fato construtivo, não linha de produto: a "
    "Meridiano não tem política comercial diferente para os dois.",
    f"segmento {segmento_dominado} com {DOMINIO_ESTILO:.1f}% de `{estilo_dominante}`",
    acao="modelo reprovado",
)

## 3.6 · O GMM — o único que entrega grau de pertencimento

O modelo de mistura gaussiana traz algo que nenhum dos outros tem: um imóvel não pertence a
um segmento, tem **probabilidade** de pertencer a cada um. Um imóvel com 0,52 no segmento A
e 0,46 no B não é um imóvel do segmento A — é um imóvel de fronteira, e tratá-lo como se
fosse do A é a origem de boa parte das ações de marketing que não funcionam.

A seleção aqui tem duas dimensões: *k* e o **tipo de covariância**, que é a forma que cada
componente pode assumir. O critério é o **BIC** — verossimilhança penalizada pela
complexidade —, e não a silhueta: o GMM é um modelo probabilístico, tem verossimilhança, e
usar uma métrica geométrica para escolher dentro da família seria trocar o critério da
família pelo de outra.

Isso precisa ser dito **antes** de ver a tabela, porque a tabela vai mostrar uma
oportunidade de trocar o critério — e ela será recusada.

In [ ]:
gmm_varredura = mist.varrer_gmm(EUCLIDIANO, K_MINIMO, K_MAXIMO, semente=SEMENTE)

# A regra declarada: dentro da família, escolhe o BIC — no k já fixado.
no_k = gmm_varredura.query("k == @K").set_index("covariancia")
COVARIANCIA = str(no_k.bic.idxmin())

fig, eixos = plt.subplots(1, 2, figsize=(15, 4.8), constrained_layout=True)
for cor, tipo in zip(config.PALETA, mist.COVARIANCIAS):
    recorte = gmm_varredura.query("covariancia == @tipo")
    eixos[0].plot(recorte.k, recorte.bic, marker="o", lw=2, color=cor, label=tipo)
eixos[0].axvline(K, color=ALERTA, ls="dashed", lw=2)
eixos[0].set_xticks(list(range(K_MINIMO, K_MAXIMO + 1)))
eixos[0].set_xlabel("k")
eixos[0].set_ylabel("BIC — menor é melhor")
eixos[0].set_title(f"BIC por covariância — no k fixado, vence `{COVARIANCIA}`")
eixos[0].legend(title="covariância")

cores = [APROVADO if v >= SILHUETA_MINIMA else ALERTA for v in no_k.silhueta]
eixos[1].bar(no_k.index, no_k.silhueta, color=cores)
eixos[1].axhline(SILHUETA_MINIMA, color=REFERENCIA, ls="dashed", lw=2)
eixos[1].bar_label(eixos[1].containers[0], fmt="%.4f", padding=3, fontsize=9)
eixos[1].set_ylabel("Silhueta")
eixos[1].set_title(f"A silhueta em k = {K} prefere outra covariância — e não é ela que decide")
eixos[1].margins(y=0.2)

salvar(fig, "gmm-selecao", gmm_varredura)

modelo_gmm, rotulos_gmm_bruto, probabilidades = mist.ajustar_gmm(
    EUCLIDIANO, K, COVARIANCIA, semente=SEMENTE
)
confianca = mist.classificar_confianca(probabilidades)

registrar(
    "3 · Construção",
    f"Fixar a covariância `{COVARIANCIA}` no GMM",
    "Seleção por BIC dentro da família",
    "O BIC é o critério próprio de um modelo probabilístico: verossimilhança penalizada "
    "pela complexidade. A covariância `diag` tem silhueta maior, e adotá-la significaria "
    "trocar o critério da família depois de ver o resultado.",
    f"BIC {no_k.loc[COVARIANCIA, 'bic']:,.0f} contra "
    f"{no_k.bic.drop(COVARIANCIA).min():,.0f} da segunda colocada".replace(",", "."),
    acao="parâmetro fixado",
)

display(no_k[["bic", "aic", "silhueta", "pct_fronteira", "convergiu"]].round(4))
print(f"Imóveis de fronteira (máx. probabilidade < 0,60): {100 * float((probabilidades.max(axis=1) < 0.60).mean()):.2f}%")
print(confianca.value_counts().to_string())

### A premissa do GMM, conferida

A covariância `diag` supõe que, **dentro de cada componente**, as variáveis não se
correlacionam. É uma suposição forte, e mensurável: basta calcular a correlação média entre
pares de variáveis dentro de cada grupo.

A medida abaixo mostra pares chegando a |ρ| perto de 1 dentro de um mesmo componente — a fase
3 já havia medido 0,81 entre `comodos` e `area_habitavel` na carteira inteira, e casa maior
continua tendo mais cômodos depois de dividida em grupos. A suposição de independência
interna está refutada pela própria base.

Isso não invalida o GMM: com covariância `full`, que é a que o BIC escolheu, cada componente
tem matriz de covariância própria e a correlação interna é modelada. Fica registrado porque
`diag` é a covariância que teria a melhor silhueta, e agora há **duas** razões independentes
para não adotá-la: o critério da família a rejeita, e a premissa que ela impõe é falsa nesta
carteira.

In [ ]:
def correlacao_interna(rotulos) -> tuple[float, float]:
    """|ρ| médio e máximo entre pares de variáveis, dentro dos grupos."""
    medias, maximos = [], []
    for grupo in np.unique(rotulos):
        matriz = np.corrcoef(EUCLIDIANO[rotulos == grupo].T)
        acima = matriz[np.triu_indices_from(matriz, k=1)]
        medias.append(float(np.nanmean(np.abs(acima))))
        maximos.append(float(np.nanmax(np.abs(acima))))
    return round(float(np.mean(medias)), 3), round(float(np.max(maximos)), 3)


premissa_gmm = pd.DataFrame(
    [
        {
            "covariancia": tipo,
            "supoe_correlacao_interna_nula": tipo in ("diag", "spherical"),
            "rho_medio_dentro_dos_grupos": correlacao_interna(
                rotular_gmm(tipo)(EUCLIDIANO, K)
            )[0],
            "rho_maximo_dentro_dos_grupos": correlacao_interna(
                rotular_gmm(tipo)(EUCLIDIANO, K)
            )[1],
        }
        for tipo in ("full", "diag")
    ]
)
io.salvar_tabela(premissa_gmm, SECAO, "gmm-premissa-de-covariancia", indice=False)
premissa_gmm

## 3.7 · A densidade — a previsão do bloco 1, conferida

O bloco 1 previu: com a nuvem unimodal e o joelho suave, o DBSCAN não encontraria estrutura —
ou tudo num grupo só, ou ruído acima do teto. A previsão é conferida varrendo `eps` em torno
do joelho, com `minPts = 2 × dimensão`, e comparando com o teto de 10% de ruído do Canvas.

O Mean Shift é a segunda tentativa: em vez de raio fixo, ele procura os **modos** da
densidade. Se houver quatro concentrações na carteira, ele as encontra sem que ninguém diga
quantas são.

In [ ]:
grade_eps = np.round(np.linspace(1.6, 4.0, 13), 3)
dbscan = den.varrer_dbscan(EUCLIDIANO, grade_eps, MIN_AMOSTRAS)

mean_shift = den.varrer_mean_shift(EUCLIDIANO, (0.2, 0.3, 0.4, 0.5))
io.salvar_tabela(mean_shift.reset_index(), SECAO, "mean-shift-varredura", indice=False)

# Uma configuração aprovada precisaria de 2 a 5 grupos E ruído dentro do teto.
aprovadas = dbscan.query(
    "grupos >= @K_MINIMO and grupos <= @K_MAXIMO and pct_ruido <= @RUIDO_MAXIMO"
)

fig, eixos = plt.subplots(1, 2, figsize=(15, 4.8), constrained_layout=True)

eixos[0].plot(dbscan.index, 100 * dbscan.pct_ruido, marker="o", lw=2, color=ALERTA,
              label="% de ruído")
eixos[0].axhline(100 * RUIDO_MAXIMO, color=REFERENCIA, ls="dashed", lw=2)
eixos[0].set_xlabel("eps")
eixos[0].set_ylabel("% da carteira como ruído")
eixos[0].set_title(f"DBSCAN — teto de ruído do Canvas em {100 * RUIDO_MAXIMO:.0f}%")
gemeo = eixos[0].twinx()
gemeo.plot(dbscan.index, dbscan.grupos, marker="s", lw=2, ls="dashed", color=BARRA,
           label="grupos")
gemeo.set_ylabel("Grupos encontrados")
gemeo.grid(False)
eixos[0].legend(loc="upper right")
gemeo.legend(loc="center right")

eixos[1].bar(mean_shift.index.astype(str), 100 * mean_shift.maior_grupo_pct, color=ALERTA)
eixos[1].bar_label(
    eixos[1].containers[0],
    labels=[f"{p:.1f}%\n{g} grupos" for p, g in
            zip(100 * mean_shift.maior_grupo_pct, mean_shift.grupos)],
    padding=3,
    fontsize=9,
)
eixos[1].set_xlabel("Quantil usado na largura de banda")
eixos[1].set_ylabel("% da carteira no maior grupo")
eixos[1].set_title("Mean Shift — um modo só, qualquer que seja a largura")
eixos[1].margins(y=0.22)

salvar(fig, "densidade-dbscan-e-mean-shift", dbscan.reset_index())

print(f"Configurações de DBSCAN que passam nos critérios: {len(aprovadas)}")
registrar(
    "3 · Construção",
    "Reprovar DBSCAN e Mean Shift",
    "Densidade — varredura de eps e de largura de banda",
    "A previsão do bloco 1 se cumpriu: a carteira é uma nuvem unimodal. Ou o eps é pequeno "
    "e quase tudo vira ruído, ou é grande e tudo vira um grupo só — não existe faixa "
    "intermediária, porque não há vale de densidade para separar.",
    f"nenhuma das {len(dbscan)} configurações de eps ficou entre {K_MINIMO} e {K_MAXIMO} "
    f"grupos com ruído ≤ {100 * RUIDO_MAXIMO:.0f}%; Mean Shift concentra "
    f"{100 * mean_shift.maior_grupo_pct.min():.1f}% no maior grupo",
    acao="modelo reprovado",
)
display(dbscan)
mean_shift

## 3.8 · Os candidatos construídos

Cinco partições sobre a mesma carteira, todas com *k* = 3. Quatro delas seguem o protocolo do
bloco 2; a quinta — o GMM com covariância `diag` — entra **explicitamente fora do
protocolo**, como o contrafactual da decisão tomada em 3.6. Ela está aqui para que o bloco 4
possa mostrar o que teria acontecido se o critério tivesse sido trocado depois de ver o
resultado, em vez de esse cenário ficar como afirmação não verificada.

Os rótulos de todos passam por `pers.ordenar_canonicamente`: sem isso, o segmento chamado
"0" muda de identidade a cada reexecução, e qualquer frase que se refira a ele fica errada na
próxima rodada.

A projeção em dois componentes principais é **só desenho**. Os dois primeiros componentes
retêm cerca de 52% da variância — o que se vê no gráfico é uma sombra de 17 dimensões, e
grupos que parecem sobrepostos ali podem estar separados nos eixos que a projeção descartou.

In [ ]:
COVARIANCIA_CONTRAFACTUAL = str(no_k.silhueta.idxmax())

rotulos_kmeans = pers.ordenar_canonicamente(base, rotular_kmeans(EUCLIDIANO, K))
rotulos_ward = pers.ordenar_canonicamente(base, rotular_ward(EUCLIDIANO, K))
rotulos_gmm = pers.ordenar_canonicamente(base, rotulos_gmm_bruto)
rotulos_contrafactual = pers.ordenar_canonicamente(
    base, rotular_gmm(COVARIANCIA_CONTRAFACTUAL)(EUCLIDIANO, K)
)

NOME_GMM = f"GMM ({COVARIANCIA})"
NOME_CONTRAFACTUAL = f"GMM ({COVARIANCIA_CONTRAFACTUAL}) · fora do protocolo"
NOME_GOWER = "K-Medoids (Gower)"

CANDIDATOS = {
    "K-Means": rotulos_kmeans,
    f"Hierárquica ({LIGACAO})": rotulos_ward,
    NOME_GMM: rotulos_gmm,
    NOME_CONTRAFACTUAL: rotulos_contrafactual,
    NOME_GOWER: rotulos_gower,
}
# O espaço em que cada candidato é medido — a regra de comparação do bloco 2.
ESPACO_DO_CANDIDATO = {nome: "euclidiano" for nome in CANDIDATOS}
ESPACO_DO_CANDIDATO[NOME_GOWER] = "Gower"

projecao = PCA(n_components=2, random_state=SEMENTE)
plano = projecao.fit_transform(EUCLIDIANO)
variancia_do_plano = 100 * projecao.explained_variance_ratio_.sum()

fig, eixos = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
for eixo, (nome, rotulos) in zip(eixos.flat, CANDIDATOS.items()):
    for segmento in np.unique(rotulos):
        marca = rotulos == segmento
        eixo.scatter(
            plano[marca, 0],
            plano[marca, 1],
            s=8,
            alpha=0.55,
            color=config.PALETA_SEGMENTOS[segmento % len(config.PALETA_SEGMENTOS)],
            label=f"{segmento} · {int(marca.sum())}",
        )
    eixo.set_title(f"{nome}\n{ESPACO_DO_CANDIDATO[nome]}", fontsize=10)
    eixo.set_xlabel("PC1")
    eixo.set_ylabel("PC2")
    eixo.legend(title="segmento", fontsize=7, title_fontsize=7, markerscale=1.6)
eixos.flat[-1].axis("off")

fig.suptitle(
    f"Os cinco candidatos, projetados nos dois primeiros componentes "
    f"({variancia_do_plano:.0f}% da variância) — projeção, não modelagem",
    fontsize=13,
)
tamanhos = pd.DataFrame(
    {nome: pd.Series(np.bincount(rotulos)) for nome, rotulos in CANDIDATOS.items()}
).T.rename(columns=lambda segmento: f"segmento {segmento}")
tamanhos["menor_pct"] = (100 * tamanhos.min(axis=1) / LINHAS).round(1)

salvar(fig, "candidatos-em-duas-dimensoes", tamanhos.reset_index(names="modelo"))
tamanhos

# 4 · Avaliação dos modelos

*Assess model* — a quarta tarefa, e a que só pode acontecer agora: o bloco 2 escreveu os
critérios, o bloco 3 construiu os candidatos, e aqui os dois se encontram sem que nenhum
número novo entre na régua.

Uma distinção do CRISP-DM que costuma se perder e vale a pena manter: *assess model* é
**avaliação técnica** — coesão, estabilidade, generalização. A avaliação de **negócio** — se
os segmentos valem o que custam, se o preço se distribui entre eles, se a diretoria consegue
operá-los — é a fase 5, com o rótulo aberto. Misturar as duas é como escolher o modelo pela
resposta que ele dá ao preço, e o projeto inteiro foi construído para não fazer isso.

Quatro exames, na ordem em que reprovam:

| Exame | Pergunta | Critério do bloco 2 |
|:---|:---|:---|
| **4.1 · Métricas internas** | Os grupos são coesos e separados? | Silhueta ≥ 0,15 · menor segmento ≥ 2% |
| **4.2 · Estabilidade** | Sobrevive a perder 20% da carteira? | ARI ≥ 0,75, incluindo no pior caso |
| **4.3 · Generalização** | Sabe classificar imóvel que nunca viu? | ARI ≥ 0,75 e silhueta ≥ 0,15 no retido |
| **4.4 · Concordância e leitura** | Os algoritmos concordam? O resultado é descritível? | Regra de desempate, itens 3 e 4 |

In [ ]:
def metricas_do_candidato(nome, rotulos):
    """Métricas internas, medidas no espaço próprio do candidato."""
    if ESPACO_DO_CANDIDATO[nome] == "Gower":
        medidas = part.metricas_internas(None, rotulos, GOWER)
    else:
        medidas = part.metricas_internas(EUCLIDIANO, rotulos)
    contagem = np.bincount(rotulos)
    return {
        "modelo": nome,
        "espaco": ESPACO_DO_CANDIDATO[nome],
        "grupos": int(len(contagem)),
        "silhueta": round(medidas["silhueta"], 4),
        "davies_bouldin": round(medidas["davies_bouldin"], 3)
        if medidas["davies_bouldin"] == medidas["davies_bouldin"]
        else np.nan,
        "menor_grupo": int(contagem.min()),
        "menor_grupo_pct": round(100 * contagem.min() / LINHAS, 1),
        "maior_grupo_pct": round(100 * contagem.max() / LINHAS, 1),
    }


internas = pd.DataFrame(
    [metricas_do_candidato(nome, rotulos) for nome, rotulos in CANDIDATOS.items()]
).set_index("modelo")

fig, ax = plt.subplots(figsize=(12, 4.8))
cores = [
    NEUTRO if ESPACO_DO_CANDIDATO[nome] == "Gower"
    else (APROVADO if valor >= SILHUETA_MINIMA else ALERTA)
    for nome, valor in internas.silhueta.items()
]
ax.barh(internas.index, internas.silhueta, color=cores)
ax.bar_label(
    ax.containers[0],
    labels=[
        f"{s:.4f}   ({e})" for s, e in zip(internas.silhueta, internas.espaco)
    ],
    padding=3,
    fontsize=9,
)
ax.axvline(SILHUETA_MINIMA, color=REFERENCIA, ls="dashed", lw=2)
ax.invert_yaxis()
ax.set_xlabel("Silhueta — medida no espaço próprio de cada candidato")
ax.set_title(
    f"Métricas internas em k = {K} · a barra cinza está em outra geometria e não se compara"
)
ax.margins(x=0.35)
salvar(fig, "metricas-internas", internas.reset_index())

internas

### O que 4.1 já resolve

Duas reprovações, e a segunda é a que interessa:

* **a hierárquica de Ward reprova por silhueta.** Ela otimiza o mesmo objetivo do K-Means —
  variância interna — e chega a uma partição pior segundo a métrica do Canvas. Não é
  surpresa: a aglomerativa toma decisões de fusão irreversíveis, enquanto o K-Means reajusta
  os centros a cada iteração. Ela cumpriu seu papel neste notebook, que era dar uma segunda
  opinião sobre o número de grupos, e a deu: a árvore concordou com a faixa de dois a quatro;
* **o GMM escolhido pelo BIC também reprova por silhueta**, com componentes que se
  sobrepõem — a mesma sobreposição que dá a ele o grau de pertencimento. E o contrafactual
  `diag`, que teria passado, é justamente o que o critério da própria família rejeitou.

O K-Means passa nos dois critérios de 4.1. O modelo de Gower passa nos critérios **no espaço
dele** — e já foi reprovado no bloco 3 pela leitura do resultado, não pela métrica.

## 4.2 e 4.3 · Estabilidade e generalização

Os dois exames que a métrica interna não faz. O primeiro pergunta se a partição se reproduz
quando a carteira muda; o segundo, se ela descreve imóvel que o modelo não viu.

Para o *hold-out*, cada modelo precisa de uma **regra de atribuição** para o conjunto
retido, e elas não são equivalentes — a diferença entre elas é, ela própria, um resultado:

* **K-Means e GMM** têm regra nativa: centroide mais próximo, probabilidade a posteriori. É
  o mesmo mecanismo que a aplicação da fase 6 usaria;
* **a hierárquica não tem.** Para incluí-la, é preciso emprestar uma: calcular o centro de
  cada grupo formado no ajuste e atribuir o retido ao mais próximo. Isso **não é** o método
  hierárquico — é um K-Means improvisado sobre os grupos dele, e o número que sai vale menos
  do que os outros da mesma coluna;
* **o K-Medoids tem regra nativa**, e boa: medoide mais próximo, com a dissimilaridade de
  Gower calculada contra imóveis reais.

In [ ]:
def atribuir_retido(nome, indices_ajuste, indices_retido):
    """Ajusta no conjunto de ajuste e atribui o retido pela regra do modelo.

    Devolve (rótulos do retido, matriz do retido, regra usada).
    """
    if nome == NOME_GOWER:
        # Gower é uma dissimilaridade global: a amplitude de cada variável é
        # propriedade da carteira, não da amostra. Recalculá-la por reamostra
        # mudaria a própria métrica, e não a estabilidade que se quer medir.
        parcial = part.kmedoids(
            GOWER[np.ix_(indices_ajuste, indices_ajuste)], K, reinicios=8, semente=SEMENTE
        )
        medoides = indices_ajuste[parcial.medoides]
        rotulos = np.argmin(GOWER[np.ix_(indices_retido, medoides)], axis=1)
        return rotulos, None, "medoide mais próximo"

    ajuste, retido = preparar(indices_ajuste, indices_retido)
    if nome.startswith("GMM"):
        tipo = COVARIANCIA if nome == NOME_GMM else COVARIANCIA_CONTRAFACTUAL
        modelo = GaussianMixture(
            n_components=K, covariance_type=tipo, n_init=5, max_iter=500, random_state=SEMENTE
        ).fit(ajuste)
        return modelo.predict(retido), retido, "probabilidade a posteriori"
    if nome == "K-Means":
        modelo = KMeans(n_clusters=K, n_init=20, random_state=SEMENTE).fit(ajuste)
        return modelo.predict(retido), retido, "centroide mais próximo"

    # Hierárquica: regra emprestada, e assinalada como tal.
    rotulos_ajuste = rotular_ward(ajuste, K)
    centros = np.vstack([ajuste[rotulos_ajuste == g].mean(axis=0) for g in np.unique(rotulos_ajuste)])
    distancia = ((retido[:, None, :] - centros[None, :, :]) ** 2).sum(axis=2)
    return distancia.argmin(axis=1), retido, "centro mais próximo (emprestada)"


def estabilidade_gower(repeticoes=REPETICOES, fracao=FRACAO, semente=SEMENTE):
    """Prova 2 para o modelo de Gower, na geometria dele."""
    aleatorio = np.random.default_rng(semente)
    referencia = part.kmedoids(GOWER, K, reinicios=8, semente=semente).rotulos
    tamanho = int(fracao * LINHAS)
    aris = []
    for _ in range(repeticoes):
        indices = np.sort(aleatorio.choice(LINHAS, size=tamanho, replace=False))
        parcial = part.kmedoids(
            GOWER[np.ix_(indices, indices)], K, reinicios=6,
            semente=int(aleatorio.integers(1e6)),
        )
        aris.append(float(adjusted_rand_score(referencia[indices], parcial.rotulos)))
    return np.array(aris)


ROTULADORES = {
    "K-Means": rotular_kmeans,
    f"Hierárquica ({LIGACAO})": rotular_ward,
    NOME_GMM: rotular_gmm(COVARIANCIA),
    NOME_CONTRAFACTUAL: rotular_gmm(COVARIANCIA_CONTRAFACTUAL),
}

linhas_provas = []
for nome in CANDIDATOS:
    aris = estabilidade_gower() if nome == NOME_GOWER else estabilidade(ROTULADORES[nome], K)
    rotulos_retido, matriz_retido, regra = atribuir_retido(nome, AJUSTE, RETIDO)
    if matriz_retido is None:
        silhueta_retido = float(
            silhouette_score(GOWER[np.ix_(RETIDO, RETIDO)], rotulos_retido, metric="precomputed")
        )
    else:
        silhueta_retido = float(silhouette_score(matriz_retido, rotulos_retido))
    linhas_provas.append(
        {
            "modelo": nome,
            "ari_medio": round(float(aris.mean()), 4),
            "ari_minimo": round(float(aris.min()), 4),
            "reamostras_abaixo_do_piso": int((aris < ARI_MINIMO).sum()),
            "regra_de_atribuicao": regra,
            "silhueta_no_retido": round(silhueta_retido, 4),
            "ari_no_retido": round(
                float(adjusted_rand_score(CANDIDATOS[nome][RETIDO], rotulos_retido)), 4
            ),
        }
    )

provas = pd.DataFrame(linhas_provas).set_index("modelo")

fig, eixos = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
posicoes = np.arange(len(provas))
eixos[0].barh(posicoes + 0.2, provas.ari_medio, height=0.38, color=BARRA, label="ARI médio")
eixos[0].barh(posicoes - 0.2, provas.ari_minimo, height=0.38, color=ALERTA,
              label="ARI mínimo (pior reamostra)")
eixos[0].axvline(ARI_MINIMO, color=REFERENCIA, ls="dashed", lw=2)
eixos[0].set_yticks(posicoes)
eixos[0].set_yticklabels(provas.index, fontsize=9)
eixos[0].invert_yaxis()
eixos[0].set_xlabel("Índice de Rand ajustado")
eixos[0].set_title(f"Prova 2 — {REPETICOES} reamostras de {int(100 * FRACAO)}%")
eixos[0].legend(loc="lower right")

eixos[1].barh(provas.index, provas.silhueta_no_retido,
              color=[APROVADO if v >= SILHUETA_MINIMA else ALERTA
                     for v in provas.silhueta_no_retido])
eixos[1].bar_label(
    eixos[1].containers[0],
    labels=[f"{s:.4f}   ARI {a:.3f}" for s, a in
            zip(provas.silhueta_no_retido, provas.ari_no_retido)],
    padding=3,
    fontsize=9,
)
eixos[1].axvline(SILHUETA_MINIMA, color=REFERENCIA, ls="dashed", lw=2)
eixos[1].invert_yaxis()
eixos[1].set_xlabel(f"Silhueta nos {pt(len(RETIDO))} imóveis retidos")
eixos[1].set_title("Prova 3 — hold-out")
eixos[1].margins(x=0.42)

salvar(fig, "provas-de-estabilidade-e-generalizacao", provas.reset_index())
provas

### O que as duas provas mostraram

A prova 2 separa os candidatos de um jeito que a silhueta não separava. Quatro leituras:

* **a hierárquica se desfaz sob reamostragem.** ARI médio de 0,43, mínimo de 0,34 — pouco
  acima do que se esperaria de duas atribuições aleatórias. Faz sentido pelo mecanismo: uma
  fusão errada no começo da árvore não é revisada, e basta um punhado de imóveis diferente
  para que a ordem das fusões mude. Ela já havia reprovado por silhueta; aqui reprova de
  novo, por outro caminho;
* **o GMM e o modelo de Gower são o caso exato que a regra do pior caso antecipou.** O GMM
  tem ARI médio de 0,91 e mínimo de 0,45; o K-Medoids sobre Gower, 0,85 e 0,30. Pela média,
  os dois passariam. Existe reamostra em que cada um deles se desfaz, e é essa reamostra que
  a próxima carga de dados pode ser — foi para pegar isto que a regra do bloco 2 leu o
  critério de estabilidade no mínimo, e não na média;
* **o contrafactual `diag` passou em todas as três provas** — silhueta 0,2015, ARI mínimo
  0,9821, *hold-out* consistente. Ele é excluído por uma razão só, e ela precisa ser dita sem
  eufemismo: **o critério da própria família o rejeitou**. O BIC escolheu `full`, e adotar
  `diag` porque a silhueta ficou melhor seria trocar a régua depois de ver o resultado. A
  premissa que ele impõe — correlação interna nula — também está refutada pela medida de 3.6.
  Se o critério tivesse sido escrito depois, este seria o modelo escolhido; é precisamente
  por isso que ele foi escrito antes;
* **o K-Means é o mais estável dos cinco no pior caso**, e a prova 3 confirma: a silhueta no
  conjunto retido (0,1782) é praticamente igual à do ajuste, e a atribuição dos 470 imóveis
  retidos concorda com o modelo ajustado na carteira inteira (ARI 0,94). Estrutura que se
  repete em dados não vistos é o mais perto de "certo" que um agrupamento chega.

Duas observações sobre a coluna do *hold-out*, para não ler mais do que ela diz. A linha da
hierárquica está medida com a regra de atribuição emprestada, e por isso a silhueta dela no
retido é boa — mas o ARI de 0,40 mostra que a partição não se reproduz. E o ARI de 1,00 do
modelo de Gower não é sinal de qualidade: atribuir pelo medoide mais próximo é a mesma
operação que definiu os grupos, então a concordância é aritmética, não evidência.

## 4.4 · Concordância entre os algoritmos

Se todos os candidatos encontrassem a mesma coisa, a escolha do algoritmo seria detalhe. A
matriz de ARI abaixo mostra que não é o caso: a concordância entre eles é **baixa**, e isso
é o resultado honesto de uma carteira sem vales de densidade. Não existe uma partição
"verdadeira" esperando ser descoberta; existem partições diferentes, defensáveis em graus
diferentes, e é por isso que o critério declarado antes é o que decide.

In [ ]:
concordancia = estab.comparar_particoes(CANDIDATOS)

contingencia = estab.contingencia(
    rotulos_kmeans, rotulos_ward, "K-Means", f"Hierárquica ({LIGACAO})"
)
io.salvar_tabela(contingencia.reset_index(), SECAO, "contingencia-kmeans-hierarquica", indice=False)

fig, eixos = plt.subplots(1, 2, figsize=(15, 5.2), constrained_layout=True)
sns.heatmap(
    concordancia.astype(float), annot=True, fmt=".3f", cmap="Blues", vmin=0, vmax=1,
    square=True, linewidths=0.6, cbar_kws={"shrink": 0.75, "label": "ARI"}, ax=eixos[0],
)
eixos[0].set_title("ARI entre os candidatos — concordância baixa")
eixos[0].tick_params(axis="x", rotation=35)
eixos[0].tick_params(axis="y", rotation=0)

sns.heatmap(
    contingencia.iloc[:-1, :-1], annot=True, fmt="d", cmap="Blues", linewidths=0.6,
    cbar=False, ax=eixos[1],
)
eixos[1].set_title(f"Onde K-Means e {LIGACAO} discordam — contingência")

salvar(fig, "concordancia-entre-candidatos", concordancia.reset_index(names="modelo"))
display(concordancia)
contingencia

## 4.5 · O placar

Os critérios do bloco 2, aplicados aos cinco candidatos, sem critério novo e sem exceção. A
coluna "leitura de negócio" não é uma régua inventada aqui: é a primeira exigência da tabela
do bloco 1 — *cada segmento vira ação comercial* —, e é ela que o modelo de Gower reprovou
em 3.5 ao separar a carteira por estilo construtivo.

In [ ]:
LEITURA_DE_NEGOCIO = {nome: True for nome in CANDIDATOS}
LEITURA_DE_NEGOCIO[NOME_GOWER] = False  # reprovado em 3.5: partição comandada por `estilo`

DENTRO_DO_PROTOCOLO = {nome: True for nome in CANDIDATOS}
DENTRO_DO_PROTOCOLO[NOME_CONTRAFACTUAL] = False  # covariância escolhida pela silhueta

placar = pd.DataFrame(index=list(CANDIDATOS))
placar["silhueta"] = internas.silhueta
placar["menor_grupo_pct"] = internas.menor_grupo_pct
placar["ari_medio"] = provas.ari_medio
placar["ari_minimo"] = provas.ari_minimo
placar["silhueta_no_retido"] = provas.silhueta_no_retido
placar["ari_no_retido"] = provas.ari_no_retido

exames = pd.DataFrame(
    {
        f"silhueta ≥ {SILHUETA_MINIMA:.2f}": placar.silhueta >= SILHUETA_MINIMA,
        f"menor grupo ≥ {100 * MENOR_SEGMENTO_MINIMO:.0f}%": (
            placar.menor_grupo_pct >= 100 * MENOR_SEGMENTO_MINIMO
        ),
        f"ARI médio ≥ {ARI_MINIMO:.2f}": placar.ari_medio >= ARI_MINIMO,
        f"ARI mínimo ≥ {ARI_MINIMO:.2f}": placar.ari_minimo >= ARI_MINIMO,
        f"hold-out: silhueta ≥ {SILHUETA_MINIMA:.2f}": (
            placar.silhueta_no_retido >= SILHUETA_MINIMA
        ),
        f"hold-out: ARI ≥ {ARI_MINIMO:.2f}": placar.ari_no_retido >= ARI_MINIMO,
        "leitura de negócio": pd.Series(LEITURA_DE_NEGOCIO),
        "dentro do protocolo": pd.Series(DENTRO_DO_PROTOCOLO),
    },
    index=placar.index,
)
placar["aprovado"] = exames.all(axis=1)
placar["criterios_reprovados"] = (~exames).sum(axis=1)

# A regra de desempate do bloco 2: entre os aprovados, o melhor pior caso; depois,
# a silhueta.
aprovados = placar[placar.aprovado].sort_values(["ari_minimo", "silhueta"], ascending=False)
CAMPEAO = str(aprovados.index[0])

fig, ax = plt.subplots(figsize=(13, 4.6))
sns.heatmap(
    exames.T.astype(int),
    annot=exames.T.map(lambda passou: "✓" if passou else "✗").to_numpy(),
    fmt="",
    cmap=plt.matplotlib.colors.ListedColormap([ALERTA, APROVADO]),
    vmin=0, vmax=1, cbar=False, linewidths=1.2, linecolor="white",
    annot_kws={"fontsize": 12, "color": "white", "fontweight": "bold"},
    ax=ax,
)
ax.set_title(f"O placar — critérios do bloco 2 aplicados aos cinco candidatos (k = {K})")
ax.tick_params(axis="x", rotation=18)
ax.tick_params(axis="y", rotation=0)
salvar(fig, "placar-dos-candidatos", placar.join(exames).reset_index(names="modelo"))

display(placar.round(4))
print(f"Aprovados em todos os critérios: {', '.join(aprovados.index)}")
print(f"Campeão pela regra de desempate: {CAMPEAO}")

## 4.6 · O modelo escolhido, e o que ele descreve

Um modelo aprovado ainda precisa passar no exame que nenhuma métrica faz: **alguém consegue
dizer o que cada grupo é?** Um agrupamento que ninguém descreve em uma frase não vira
política comercial, e a fase 5 herdaria um problema em vez de um resultado.

Duas tabelas respondem. O **perfil** traz a mediana de cada atributo por segmento, em pés
quadrados e anos — as unidades que a fase 3 preservou justamente para esta hora. A
**assinatura** mostra o quanto cada segmento se afasta da carteira inteira, em desvios da
mediana global, e é ela que dá o nome de negócio a cada eixo.

O representante real fecha a leitura: para cada segmento, o imóvel da carteira mais próximo
do centro do grupo — com identificador, porque a diretoria vai querer ver um.

In [ ]:
ROTULOS_FINAIS = CANDIDATOS[CAMPEAO]

perfil = perf.perfil_por_segmento(base, ROTULOS_FINAIS, ESPACO_EUCLIDIANO)
assinatura = perf.assinatura(base, ROTULOS_FINAIS, ESPACO_EUCLIDIANO)
distintivos = perf.atributos_distintivos(assinatura, quantos=4)
silhueta_por_grupo = part.silhueta_por_grupo(EUCLIDIANO, ROTULOS_FINAIS)

io.salvar_tabela(assinatura.reset_index(), SECAO, "assinatura-dos-segmentos", indice=False)
io.salvar_tabela(
    silhueta_por_grupo.reset_index(), SECAO, "silhueta-por-segmento", indice=False
)

fig, eixos = plt.subplots(1, 2, figsize=(16, 6.5),
                          gridspec_kw={"width_ratios": [2.1, 1]}, constrained_layout=True)
sns.heatmap(
    assinatura.T.rename(index=dic.rotulo),
    annot=True, fmt=".2f", cmap="RdBu_r", center=0, linewidths=0.6,
    cbar_kws={"shrink": 0.8, "label": "desvios da mediana da carteira"}, ax=eixos[0],
)
eixos[0].set_title("Assinatura dos segmentos — o que cada um tem de diferente")
eixos[0].set_xlabel("Segmento")
eixos[0].set_ylabel("")

cores = [config.PALETA_SEGMENTOS[g % len(config.PALETA_SEGMENTOS)]
         for g in silhueta_por_grupo.index]
eixos[1].barh(silhueta_por_grupo.index.astype(str), silhueta_por_grupo.media, color=cores)
eixos[1].bar_label(
    eixos[1].containers[0],
    labels=[f"{m:.3f}   {int(r)} imóveis   {100 * n:.1f}% neg."
            for m, r, n in zip(silhueta_por_grupo.media, silhueta_por_grupo.registros,
                               silhueta_por_grupo.pct_negativa)],
    padding=3, fontsize=8,
)
eixos[1].axvline(SILHUETA_MINIMA, color=REFERENCIA, ls="dashed", lw=2)
eixos[1].invert_yaxis()
eixos[1].set_xlabel("Silhueta média do segmento")
eixos[1].set_title("A média global esconde o grupo esgarçado")
eixos[1].margins(x=0.75)

fig.suptitle(f"{CAMPEAO} · k = {K} — o que os segmentos são", fontsize=13)
salvar(fig, "perfil-do-modelo-escolhido", perfil.reset_index())

print(f"Modelo: {CAMPEAO} · k = {K}\n")
for segmento, atributos in distintivos.items():
    tamanho = int((ROTULOS_FINAIS == segmento).sum())
    print(f"Segmento {segmento} — {pt(tamanho)} imóveis ({100 * tamanho / LINHAS:.1f}%)")
    for atributo, valor in atributos:
        print(f"    {atributo:<26} {valor:+.2f}")
    print()

display(perfil.T)
assinatura.T

In [ ]:
# O representante real de cada segmento: o imóvel da carteira mais próximo do
# centro do grupo. O centroide do K-Means é uma média que pode ter 2,7 dormitórios.
distancias_no_espaco = dist.matriz_distancia(EUCLIDIANO)
representantes = []
for segmento in np.unique(ROTULOS_FINAIS):
    indices = np.flatnonzero(ROTULOS_FINAIS == segmento)
    interno = distancias_no_espaco[np.ix_(indices, indices)].sum(axis=1)
    registro = base.iloc[indices[int(np.argmin(interno))]]
    representantes.append(
        {
            "segmento": int(segmento),
            "pid": int(registro.pid),
            "area_habitavel": int(registro.area_habitavel),
            "qualidade_geral": int(registro.qualidade_geral),
            "idade_imovel": int(registro.idade_imovel),
            "banheiros_total": float(registro.banheiros_total),
            "comodos": int(registro.comodos),
            "estilo": registro.estilo,
            "tipo_construcao": registro.tipo_construcao,
        }
    )
representantes = pd.DataFrame(representantes).set_index("segmento")
io.salvar_tabela(representantes.reset_index(), SECAO, "representantes-reais", indice=False)

registrar(
    "4 · Avaliação",
    f"Reprovar a hierárquica ({LIGACAO})",
    "Métricas internas e prova 2",
    "Otimiza o mesmo objetivo do K-Means e chega a uma partição pior: as fusões são "
    "irreversíveis, e basta um punhado de imóveis diferente para mudar a ordem delas.",
    f"silhueta {placar.loc[f'Hierárquica ({LIGACAO})', 'silhueta']:.4f} · "
    f"ARI médio {placar.loc[f'Hierárquica ({LIGACAO})', 'ari_medio']:.4f}",
    acao="modelo reprovado",
)
registrar(
    "4 · Avaliação",
    f"Reprovar o {NOME_GMM}",
    "Métricas internas e prova 2 no pior caso",
    "Os componentes se sobrepõem — a mesma sobreposição que dá ao GMM o grau de "
    "pertencimento derruba a separação medida pela silhueta. E a estabilidade alta na "
    "média esconde reamostras em que a partição se desfaz.",
    f"silhueta {placar.loc[NOME_GMM, 'silhueta']:.4f} contra o piso de {SILHUETA_MINIMA:.2f} · "
    f"ARI médio {placar.loc[NOME_GMM, 'ari_medio']:.4f}, "
    f"mínimo {placar.loc[NOME_GMM, 'ari_minimo']:.4f}",
    acao="modelo reprovado",
)
registrar(
    "4 · Avaliação",
    f"Reprovar o {NOME_GOWER} também na prova 2",
    "Prova 2 no pior caso",
    "Já reprovado em 3.5 pela leitura de negócio, ele reprova também por estabilidade: a "
    "média esconde reamostras em que a partição se desfaz. Duas razões independentes.",
    f"ARI médio {placar.loc[NOME_GOWER, 'ari_medio']:.4f}, "
    f"mínimo {placar.loc[NOME_GOWER, 'ari_minimo']:.4f}",
    acao="modelo reprovado",
)
registrar(
    "4 · Avaliação",
    f"Não adotar o contrafactual `{COVARIANCIA_CONTRAFACTUAL}`, apesar de aprovado nas provas",
    "Coerência do critério de seleção",
    "Ele passa nas três provas — e continua fora porque o critério da própria família o "
    "rejeitou: o BIC escolheu outra covariância, e adotar esta por causa da silhueta seria "
    "trocar a régua depois de ver o resultado. A premissa de correlação interna nula que "
    "ela impõe também está refutada pela medida de 3.6.",
    f"silhueta {placar.loc[NOME_CONTRAFACTUAL, 'silhueta']:.4f} e ARI mínimo "
    f"{placar.loc[NOME_CONTRAFACTUAL, 'ari_minimo']:.4f}, com BIC "
    f"{pt(int(no_k.loc[COVARIANCIA_CONTRAFACTUAL, 'bic'] - no_k.loc[COVARIANCIA, 'bic']))} "
    "acima do escolhido",
    acao="modelo reprovado",
)
registrar(
    "4 · Avaliação",
    f"Escolher {CAMPEAO} com k = {K}",
    "Regra de desempate do bloco 2",
    "Único candidato aprovado em todos os critérios declarados antes de rodar: coeso "
    "acima do piso, estável no pior caso, com estrutura que se repete em imóveis não "
    "vistos e segmentos descritíveis em unidades de negócio.",
    f"silhueta {placar.loc[CAMPEAO, 'silhueta']:.4f} · ARI mínimo "
    f"{placar.loc[CAMPEAO, 'ari_minimo']:.4f} · hold-out silhueta "
    f"{placar.loc[CAMPEAO, 'silhueta_no_retido']:.4f} / ARI "
    f"{placar.loc[CAMPEAO, 'ari_no_retido']:.4f}",
    acao="modelo escolhido",
)

representantes

# 5 · O que a fase 5 recebe

Três artefatos, e a diferença em relação à fase 3 é deliberada: aqui o modelo vai para o
disco **ajustado**.

O motivo é o mesmo que levou a fase 3 a gravar a receita crua, lido do outro lado. Lá, o
ajuste sobre a carteira inteira contaminaria a validação — e a validação ainda não tinha
acontecido. Aqui ela já aconteceu, com o protocolo do bloco 2, e o que se grava é o modelo
de produção: a receita e o agrupador no mesmo objeto, ajustados sobre os 2.350 imóveis, para
que nenhuma camada da aplicação reimplemente a padronização por conta própria.

Junto com o modelo vai o **mapa canônico**. Os rótulos do K-Means não têm ordem — rodar de
novo com outra semente troca o número dos grupos —, e a fase 5 vai escrever texto sobre
"o segmento 0". O mapa é o que garante que o segmento 0 continue sendo o mesmo imóvel
compacto e antigo na próxima execução.

| Artefato | Conteúdo |
|:---|:---|
| `models/modelo-segmentacao.joblib` | Receita + agrupador ajustados, mapa canônico e metadados |
| `data/processed/base_segmentada.parquet` | Identificador, segmento, silhueta individual e distância ao centro — **sem preço** |
| `data/processed/manifesto_modelagem.json` | k, algoritmo, critérios, medidas e o critério que ficou pendente |

In [ ]:
config.garantir_diretorios()

ESTIMADORES_FINAIS = {
    "K-Means": lambda: KMeans(n_clusters=K, init="k-means++", n_init=20, random_state=SEMENTE),
    NOME_GMM: lambda: GaussianMixture(
        n_components=K, covariance_type=COVARIANCIA, n_init=10, max_iter=500, random_state=SEMENTE
    ),
    NOME_CONTRAFACTUAL: lambda: GaussianMixture(
        n_components=K, covariance_type=COVARIANCIA_CONTRAFACTUAL, n_init=10,
        max_iter=500, random_state=SEMENTE,
    ),
}
if CAMPEAO not in ESTIMADORES_FINAIS:
    raise ValueError(
        f"{CAMPEAO} não tem regra nativa de atribuição para imóvel novo e não pode ser "
        "serializado como modelo de produção — ver bloco 4.2."
    )

modelo_final = Pipeline(
    [("preparo", clone(preprocessador)), ("segmentacao", ESTIMADORES_FINAIS[CAMPEAO]())]
).fit(base[ESPACO_EUCLIDIANO])

# Os rótulos crus do estimador não têm ordem; o mapa canônico traduz para a
# numeração estável usada em toda a documentação.
rotulos_crus = modelo_final.predict(base[ESPACO_EUCLIDIANO])
MAPA_CANONICO = {
    int(cru): int(pd.Series(ROTULOS_FINAIS[rotulos_crus == cru]).mode().iloc[0])
    for cru in np.unique(rotulos_crus)
}
traduzidos = np.array([MAPA_CANONICO[int(cru)] for cru in rotulos_crus])

assert sorted(MAPA_CANONICO.values()) == list(range(K)), "o mapa canônico não é uma bijeção"
assert (traduzidos == ROTULOS_FINAIS).all(), "o modelo gravado não reproduz a partição avaliada"

amostras_silhueta = part.silhueta_por_grupo(EUCLIDIANO, ROTULOS_FINAIS)
centros = np.vstack([EUCLIDIANO[ROTULOS_FINAIS == g].mean(axis=0) for g in range(K)])
distancia_ao_centro = np.linalg.norm(EUCLIDIANO - centros[ROTULOS_FINAIS], axis=1)

base_segmentada = base[IDENTIFICADORES].copy()
base_segmentada["segmento"] = ROTULOS_FINAIS
base_segmentada["distancia_ao_centro"] = distancia_ao_centro.round(4)
base_segmentada["silhueta"] = np.round(silhouette_samples(EUCLIDIANO, ROTULOS_FINAIS), 4)

assert not PROIBIDAS & set(base_segmentada.columns), "o rótulo vazou para a base segmentada"
assert len(base_segmentada) == LINHAS, "o notebook não pode remover registro"
assert base_segmentada.pid.is_unique, "o identificador precisa ser único"

ARQUIVO_MODELO = config.MODELS / "modelo-segmentacao.joblib"
ARQUIVO_SEGMENTADA = config.DATA_PROCESSED / "base_segmentada.parquet"

dump(
    {
        "pipeline": modelo_final,
        "mapa_canonico": MAPA_CANONICO,
        "algoritmo": CAMPEAO,
        "k": K,
        "espaco": ESPACO_EUCLIDIANO,
        "semente": SEMENTE,
    },
    ARQUIVO_MODELO,
)
base_segmentada.to_parquet(ARQUIVO_SEGMENTADA, index=False)

registro = pd.DataFrame(DECISOES)
io.salvar_tabela(registro, SECAO, "registro-de-decisoes", indice=False)

manifesto = {
    "origem": BASE_MODELAGEM.relative_to(RAIZ).as_posix(),
    "modelo": ARQUIVO_MODELO.relative_to(RAIZ).as_posix(),
    "base_segmentada": ARQUIVO_SEGMENTADA.relative_to(RAIZ).as_posix(),
    "algoritmo": CAMPEAO,
    "k": K,
    "espaco": "euclidiano",
    "variaveis": ESPACO_EUCLIDIANO,
    "tamanhos": np.bincount(ROTULOS_FINAIS).tolist(),
    "mapa_canonico": MAPA_CANONICO,
    "silhueta": float(placar.loc[CAMPEAO, "silhueta"]),
    "ari_reamostragem_medio": float(placar.loc[CAMPEAO, "ari_medio"]),
    "ari_reamostragem_minimo": float(placar.loc[CAMPEAO, "ari_minimo"]),
    "silhueta_no_retido": float(placar.loc[CAMPEAO, "silhueta_no_retido"]),
    "ari_no_retido": float(placar.loc[CAMPEAO, "ari_no_retido"]),
    "criterios": {
        "silhueta_minima": SILHUETA_MINIMA,
        "k_minimo": K_MINIMO,
        "k_maximo": K_MAXIMO,
        "ari_minimo": ARI_MINIMO,
        "menor_segmento_minimo": MENOR_SEGMENTO_MINIMO,
        "ruido_maximo": RUIDO_MAXIMO,
    },
    "criterio_pendente": "ARI contra quartis de preço < 0,40 — fase 5",
    "modelos_reprovados": sorted(placar.index[~placar.aprovado].tolist()),
    "execucoes_da_grade": int(grade.execucoes.iloc[-1]),
    "reamostras_por_configuracao": REPETICOES,
    "fracao_de_ajuste": FRACAO,
    "semente": SEMENTE,
    "decisoes": int(len(registro)),
    "decisoes_por_acao": registro.acao.value_counts().to_dict(),
}
io.salvar_json(manifesto, config.DATA_PROCESSED / "manifesto_modelagem.json")

print(f"Modelo gravado em {ARQUIVO_MODELO.relative_to(RAIZ)}")
print(f"Base segmentada em {ARQUIVO_SEGMENTADA.relative_to(RAIZ)}")
print(f"{CAMPEAO} · k = {K} · tamanhos {np.bincount(ROTULOS_FINAIS).tolist()} · "
      f"{len(registro)} decisões")
base_segmentada.head(8)

In [ ]:
registro

# Próximo passo

A fase 4 termina com um modelo escolhido por critério declarado antes do resultado, e a
fase 5 recebe três coisas:

* **`models/modelo-segmentacao.joblib`** — receita e agrupador ajustados no mesmo objeto,
  com o mapa canônico dos segmentos e a semente;
* **`data/processed/base_segmentada.parquet`** — 2.350 imóveis com segmento, silhueta
  individual e distância ao centro. Sem preço, sem bairro: quem fizer a leitura de negócio
  precisa juntar este arquivo à base curada por `pid`, e esse *join* é o momento — declarado
  e visível — em que o rótulo é aberto;
* **`data/processed/manifesto_modelagem.json`** — o que foi decidido, com que medida, e o
  que ficou pendente.

A primeira tarefa da fase 5 é o **critério pendente**: ARI da segmentação contra os quartis
de `preco_venda`, com piso de 0,40. Se der acima disso, a segmentação é uma tabela de preço
com outro nome, e a decisão deste notebook volta para revisão — o registro mostra
exatamente onde mexer. Depois vêm as personas, a ação por segmento e a métrica de
acompanhamento, que é o que transforma três grupos em três políticas.

## O que voltar para o código

Quatro divergências entre o que este notebook mediu e o que o repositório declara. Todas são
revisão de código ou de documentação, e nenhuma é improviso de notebook:

* **`src/model/personas.py` descreve quatro personas.** O catálogo foi escrito para a
  segmentação anterior. Com *k* = 3, ele precisa ser reescrito na fase 5 — e `PERSONAS` só
  deve ser tocado depois que o critério pendente passar, senão o texto de negócio nasce
  amarrado a um modelo que ainda pode voltar;
* **a documentação publicada responde "4" à pergunta SMART 1** (`docs/pbl/canvas-do-problema.md`,
  `docs/criterios-sucesso.md`, `docs/modelagem.md`). Aquele número foi medido sobre 2.925
  imóveis e 21 variáveis, antes da depuração da carteira e da remoção das binárias. Não é o
  mesmo experimento, e as páginas precisam registrar os dois — o resultado antigo e o atual,
  com a base de cada um;
* **`src.model.estabilidade.estabilidade_bootstrap` só serve para K-Means** e opera sobre uma
  matriz já padronizada. Este notebook precisou generalizar nas duas pontas: rotulador como
  parâmetro, para comparar famílias, e padronização dentro da reamostra, para não vazar. A
  função do módulo deve absorver as duas mudanças;
* **`config.SILHUETA_MINIMA` fez trabalho pesado aqui.** Foi ele que reprovou *k* = 4 por
  0,0112 de diferença. O Canvas justifica o limiar em palavras — "piso abaixo do qual os
  grupos não têm separação defensável" —, mas não diz de onde vem o **0,15**. Um número que
  decide sozinho quantas linhas de produto a empresa vai operar precisa de derivação, não só
  de definição: qual separação mínima a diretoria considera acionável, e medida em quê.

O padrão se repete desde o notebook 02: **quando a medida diverge da declaração, é a
declaração que se revisa, com o número na mão.**